# Zooming in after Haiyan: reliability-qualified nighttime lights across Samar–Leyte

The regional Samar–Leyte signal establishes that reliability-qualified NTL can follow broad electricity-system behaviour when observability is enforced. This notebook now asks what that signal looks like locally.

The story proceeds from space to time:

1. locate the selected cities, municipalities, and characteristic POIs;
2. map baseline nighttime lights and stage-specific recovery;
3. inspect each POI together with its spatial completeness and time series;
4. compare municipality trajectories in one view; and
5. summarise impact, T50, T80, and T90 visually before reporting exact values.

`DNB_BRDF_Corrected_NTL` is the main signal. `MQF == 0`, GHSL G2, and the declared spatial-completeness gate define admissible observations. Gap-filled NTL remains diagnostic only.

> **Interpretive boundary.** Regional NGCP agreement supports local analysis as a nested test of electricity-dependent nocturnal activity. It does not validate municipal electricity restoration. Local claims remain conditional on observability and on construct-matched evidence from grid, daytime physical-recovery, housing, and relocation studies.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import Point

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ============================================================
# 2. PATHS AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
NGCP_CSV_PATH = DATA_DIR / "ngcp" / "NGCP_Hourly_Demand.csv"


def find_dataset(patterns, label):
    '''Return the first unique match and show all candidates.'''

    matches = []

    for pattern in patterns:
        matches.extend(DATA_DIR.glob(pattern))

    matches = sorted({path.resolve() for path in matches})

    if not matches:
        raise FileNotFoundError(
            f"No {label} matched under {DATA_DIR}.\n"
            f"Patterns: {patterns}"
        )

    if len(matches) > 1:
        print(f"{label}: multiple matches; using {matches[0]}")
        for candidate in matches:
            print("  ", candidate)

    return matches[0]


GHSL_PATH = find_dataset(
    [
        "VNP46/GHSL_SMOD_E2015.tif",
        "ghsl/GHSL_SMOD_E2015.tif",
        "**/*GHSL*SMOD*.tif",
    ],
    "GHSL SMOD raster",
)

MUNICITIES_PATH = find_dataset(
    [
        "**/*MuniCities*.shp",
        "**/*municities*.shp",
        "**/*Muni*Cit*.shp",
        "**/*Municipal*.shp",
    ],
    "MuniCities shapefile",
)

ROADS_PATH = find_dataset(
    [
        "**/*Roads*.shp",
        "**/*roads*.shp",
        "**/*Road*.shp",
    ],
    "Roads shapefile",
)

OUTPUT_DIR = PROJECT_DIR / "output" / "municity_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Event and temporal design
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS = 60
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)

STAGE_WINDOWS = {
    "Baseline": (BASELINE_START, PRE_EVENT_END),
    "Stage 1 (0–59 d)": (
        EVENT_DATE,
        EVENT_DATE + pd.Timedelta(days=59),
    ),
    "Stage 2 (60–119 d)": (
        EVENT_DATE + pd.Timedelta(days=60),
        EVENT_DATE + pd.Timedelta(days=119),
    ),
    "Stage 3 (120–179 d)": (
        EVENT_DATE + pd.Timedelta(days=120),
        EVENT_DATE + pd.Timedelta(days=179),
    ),
    "Extended (180–363 d)": (
        EVENT_DATE + pd.Timedelta(days=180),
        PROFILE_END,
    ),
}

# VNP46A2 layers
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")

# RQ1 configurations. G2 is primary; change to G3 when a
# municipality lacks enough dense-urban support.
GHSL_MASKS = {
    "G2": (23, 30),
    "G3": (22, 23, 30),
    "G4": (21, 22, 23, 30),
}
SETTLEMENT_MASK = "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0

MIN_BASELINE_OBSERVATIONS = {
    1: 5,
    4: 3,
}

# Metric admissibility and persistence rules
PERSISTENCE_BLOCKS = 2
MIN_EVENT_RETENTION_PCT = 20.0
MIN_EVENT_COMPOSITES = 8
MAX_INTERPRETABLE_GAP_DAYS = 24

# Local targets: severe east-coast/landfall units, an inland
# contrast, and western urban comparisons. These are not
# unaffected controls.
TARGET_UNITS = [
    "Tacloban City",
    "Palo",
    "Tanauan",
    "Tolosa",
    "Dulag",
    "Alangalang",
    "Basey",
    "Guiuan",
    "Ormoc City",
    "Baybay City",
    "Borongan City",
    "Calbayog City",
]

FOCUS_UNIT = "Tacloban City"
POI_KERNEL_SIZE = 5

KNOWN_FILL_VALUES = (-9999.0, -32768.0, 6553.5, 65535.0)

print("Project:", PROJECT_DIR)
print("MuniCities:", MUNICITIES_PATH)
print("Roads:", ROADS_PATH)
print("Primary GHSL mask:", SETTLEMENT_MASK, GHSL_MASKS[SETTLEMENT_MASK])
print("Spatial-completeness gate:", f"{SPATIAL_COMPLETENESS_PCT:.0f}%")

In [ ]:
# ============================================================
# 3. EVIDENCE MATRIX
# ============================================================

evidence_matrix = pd.DataFrame(
    [
        {
            "unit": "Tacloban City",
            "documented_behaviour": (
                "Catastrophic storm-surge and wind damage; severe distribution-system "
                "damage; central rebuilding and northern relocation produced spatially "
                "different recovery processes."
            ),
            "NTL_profile_to_test": (
                "Large observable shock if cloud gaps permit; central functional rebound "
                "may precede housing and relocation outcomes; later spatial redistribution "
                "is possible."
            ),
            "construct_boundary": (
                "Agreement with physical rebuilding does not establish household recovery; "
                "new northern lights may indicate relocation rather than return."
            ),
            "sources": (
                "Sheykhmousa et al. 2019; Ghaffarian et al. 2021; "
                "JICA 2015; Palagi & Javernick-Will 2018"
            ),
        },
        {
            "unit": "Palo / Tanauan / Tolosa / Dulag",
            "documented_behaviour": (
                "Severely affected eastern Leyte corridor; Tanauan field surveys document "
                "extreme storm-surge impacts; DORELCO distribution facilities were heavily damaged."
            ),
            "NTL_profile_to_test": (
                "Strong shock and staged recovery are plausible, but small G2 support may "
                "make G3 and interval timing more defensible."
            ),
            "construct_boundary": (
                "Municipality-wide radiance cannot identify household service or feeder sequence."
            ),
            "sources": "Yi et al. 2015; JICA 2015; Ghaffarian et al. 2020",
        },
        {
            "unit": "Guiuan",
            "documented_behaviour": (
                "Near Haiyan's first landfall; severe damage and Eastern Samar distribution-system loss."
            ),
            "NTL_profile_to_test": (
                "Large shock is plausible, but low-light support and post-landfall cloud may "
                "make magnitude or timing unidentifiable."
            ),
            "construct_boundary": "A missing or weak curve is not evidence of limited impact.",
            "sources": "NDRRMC 2014; JICA 2015",
        },
        {
            "unit": "Basey",
            "documented_behaviour": (
                "Storm-surge-exposed Samar coast within a heavily damaged distribution-service region."
            ),
            "NTL_profile_to_test": (
                "Disruption is plausible; observability may dominate because the illuminated support is small."
            ),
            "construct_boundary": "Prefer not observable over not recovered when support is inadequate.",
            "sources": "NDRRMC 2014; JICA 2015",
        },
        {
            "unit": "Alangalang",
            "documented_behaviour": (
                "Inland municipal centre used as a morphology and exposure contrast to the coastal corridor."
            ),
            "NTL_profile_to_test": (
                "A smaller or shorter shock than exposed coastal units is plausible, but not assumed."
            ),
            "construct_boundary": "This is a contrast, not an unaffected control.",
            "sources": "Principe et al. 2026; NDRRMC 2014",
        },
        {
            "unit": "Ormoc City / Baybay City",
            "documented_behaviour": (
                "Western Leyte urban hubs with persistent lights and different exposure from Leyte Gulf."
            ),
            "NTL_profile_to_test": (
                "Smaller shock or earlier functional return may occur, while regional grid disruption "
                "can still affect both cities."
            ),
            "construct_boundary": "Do not label either city unaffected without hazard and utility evidence.",
            "sources": "Principe et al. 2026; Arroyo & Åstrand 2019",
        },
    ]
)

In [ ]:
# ============================================================
# 4. VNP46A2 HELPERS
# ============================================================


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(f"No `date` variable found. Variables: {list(ds.variables)}")

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]
    dates = pd.DatetimeIndex(pd.to_datetime(ds["date"].values)).normalize()
    ds = ds.assign_coords(date=(observation_dim, dates.values))

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        attrs = ds["spatial_ref"].attrs
        stored_crs = attrs.get("crs_wkt") or attrs.get("spatial_ref")

        if stored_crs is not None:
            ds = ds.rio.write_crs(stored_crs, inplace=False)

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        if (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        ):
            ds = ds.rio.write_crs("EPSG:4326", inplace=False)
        else:
            raise ValueError("The VNP46A2 CRS could not be recovered.")

    return ds


def clean_radiance(values):
    cleaned = values.astype("float32").where(np.isfinite(values))
    fill_values = list(KNOWN_FILL_VALUES)

    for source in (values.attrs, values.encoding):
        for key in ("_FillValue", "missing_value"):
            if source.get(key) is not None:
                fill_values.append(source[key])

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)
            if np.isfinite(fill_value):
                cleaned = cleaned.where(~np.isclose(cleaned, fill_value))
        except (TypeError, ValueError):
            continue

    return cleaned.where(cleaned >= 0)


if not A2_ZARR_PATH.exists():
    raise FileNotFoundError(f"VNP46A2 Zarr not found:\n{A2_ZARR_PATH}")

a2 = prepare_spatial_metadata(
    standardise_date_dimension(open_zarr_safely(A2_ZARR_PATH))
).sel(date=slice(ANALYSIS_START, PROFILE_END))

required_bands = [DNB_BAND, MQF_BAND]
missing_bands = [band for band in required_bands if band not in a2.data_vars]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available: {list(a2.data_vars)}"
    )

dnb = clean_radiance(a2[DNB_BAND])
mqf = a2[MQF_BAND]
dnb, mqf = xr.align(dnb, mqf, join="inner")

gap_filled = (
    clean_radiance(a2[GAP_FILLED_BAND])
    if GAP_FILLED_BAND in a2.data_vars
    else None
)

print("A2 dimensions:", dict(a2.sizes))
print("A2 CRS:", a2.rio.crs)
print("Available dates:", pd.Timestamp(a2.date.min().item()).date(), "to", pd.Timestamp(a2.date.max().item()).date())

In [ ]:
# ============================================================
# 5. LOAD MUNICIPALITIES AND ROADS; DISCOVER FIELD NAMES
# ============================================================


def normalise_column_name(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def resolve_column(gdf, candidates, label):
    lookup = {normalise_column_name(column): column for column in gdf.columns}

    for candidate in candidates:
        key = normalise_column_name(candidate)
        if key in lookup:
            return lookup[key]

    for candidate in candidates:
        key = normalise_column_name(candidate)
        for normalised, column in lookup.items():
            if key in normalised or normalised in key:
                return column

    raise KeyError(
        f"Could not identify the {label} field. "
        f"Available columns: {list(gdf.columns)}"
    )


def canonical_unit_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(character for character in text if not unicodedata.combining(character))
    tokens = re.sub(r"[^A-Z0-9]+", " ", text.upper()).split()
    stop_words = {"CITY", "OF", "MUNICIPALITY", "MUNICIPAL", "MUN"}
    return " ".join(token for token in tokens if token not in stop_words)


municipalities = gpd.read_file(MUNICITIES_PATH)

if municipalities.crs is None:
    raise ValueError("MuniCities shapefile does not contain a CRS.")

municipalities = municipalities.loc[
    municipalities.geometry.notna() & ~municipalities.geometry.is_empty
].copy()

try:
    municipalities["geometry"] = municipalities.geometry.make_valid()
except AttributeError:
    municipalities["geometry"] = municipalities.geometry.buffer(0)

NAME_COLUMN = resolve_column(
    municipalities,
    [
        "ADM3_EN",
        "ADM3_NAME",
        "MuniCity",
        "Muni_City",
        "CITY_MUN",
        "NAME_2",
        "NAME_3",
        "LGU_NAME",
        "MUNICIPALITY",
        "NAME",
    ],
    "city/municipality name",
)

province_candidates = [
    "ADM2_EN",
    "ADM2_NAME",
    "PROVINCE",
    "PROV_NAME",
    "NAME_1",
]

try:
    PROVINCE_COLUMN = resolve_column(
        municipalities,
        province_candidates,
        "province name",
    )
except KeyError:
    PROVINCE_COLUMN = None

municipalities["unit_name"] = municipalities[NAME_COLUMN].astype(str).str.strip()
municipalities["unit_key"] = municipalities["unit_name"].map(canonical_unit_name)
municipalities["province_name"] = (
    municipalities[PROVINCE_COLUMN].astype(str).str.strip()
    if PROVINCE_COLUMN is not None
    else "Not supplied"
)

# Exact canonical matching first; conservative substring matching second.
selected_rows = []
unmatched_targets = []

for target in TARGET_UNITS:
    target_key = canonical_unit_name(target)
    exact = municipalities.loc[municipalities["unit_key"] == target_key]

    if len(exact) == 1:
        selected_rows.append(exact.index[0])
        continue

    partial = municipalities.loc[
        municipalities["unit_key"].str.contains(target_key, regex=False)
        | pd.Series(
            [target_key in key for key in municipalities["unit_key"]],
            index=municipalities.index,
        )
    ]

    if len(partial) == 1:
        selected_rows.append(partial.index[0])
    else:
        unmatched_targets.append(target)

selected_municipalities = municipalities.loc[
    list(dict.fromkeys(selected_rows))
].copy()

if selected_municipalities.empty:
    raise ValueError(
        "None of TARGET_UNITS matched the MuniCities shapefile. "
        "Inspect the available names printed below and edit TARGET_UNITS."
    )

selected_municipalities["profile_id"] = np.arange(
    1,
    len(selected_municipalities) + 1,
)

roads = gpd.read_file(ROADS_PATH)

if roads.crs is None:
    raise ValueError("Roads shapefile does not contain a CRS.")

roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

print("Name field:", NAME_COLUMN)
print("Province field:", PROVINCE_COLUMN)
print("Matched targets:")
display(selected_municipalities[["unit_name", "province_name", "unit_key"]])

if unmatched_targets:
    print("Unmatched targets; edit TARGET_UNITS if needed:", unmatched_targets)

print("First 30 available municipality names:")
display(
    municipalities[["unit_name", "province_name"]]
    .sort_values(["province_name", "unit_name"])
    .head(30)
)

In [ ]:
# ============================================================
# 6. ALIGN GHSL AND RASTERIZE MUNICIPAL SUPPORT
# ============================================================

ghsl = rxr.open_rasterio(GHSL_PATH, masked=True)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(band=0, drop=True)

if ghsl.rio.crs is None:
    raise ValueError("The GHSL raster does not contain a CRS.")

viirs_template = dnb.isel(date=0, drop=True)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
).assign_coords(x=viirs_template["x"], y=viirs_template["y"])

ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK]).fillna(False)

municipalities_raster_crs = municipalities.to_crs(a2.rio.crs)
selected_raster_crs = selected_municipalities.to_crs(a2.rio.crs)
roads_raster_crs = roads.to_crs(a2.rio.crs)

raster_bounds = dnb.rio.bounds()
municipalities_raster_crs = municipalities_raster_crs.cx[
    raster_bounds[0]:raster_bounds[2],
    raster_bounds[1]:raster_bounds[3],
].copy()


def rasterize_units(gdf, value_column):
    shapes = [
        (geometry, int(value))
        for geometry, value in zip(gdf.geometry, gdf[value_column])
        if geometry is not None and not geometry.is_empty
    ]

    values = rasterize(
        shapes,
        out_shape=(dnb.sizes["y"], dnb.sizes["x"]),
        transform=dnb.rio.transform(recalc=True),
        fill=0,
        all_touched=False,
        dtype="int32",
    )

    return xr.DataArray(
        values,
        dims=SPATIAL_DIMS,
        coords={"y": dnb["y"], "x": dnb["x"]},
    )


municipalities_raster_crs = municipalities_raster_crs.copy()
municipalities_raster_crs["all_unit_id"] = np.arange(
    1,
    len(municipalities_raster_crs) + 1,
)

all_zone_id = rasterize_units(municipalities_raster_crs, "all_unit_id")
selected_zone_id = rasterize_units(selected_raster_crs, "profile_id")

study_mask = all_zone_id > 0
rq_base_mask = (ghsl_mask & study_mask).compute()

print("GHSL classes:", GHSL_MASKS[SETTLEMENT_MASK])
print("RQ settlement pixels within MuniCities:", f"{int(rq_base_mask.sum().item()):,}")
print("Any selected LGU represented on the VIIRS grid:", bool((selected_zone_id > 0).any().item()))

In [ ]:
# ============================================================
# 7. BUILD THE RELIABILITY-QUALIFIED CUBE
# ============================================================

rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_quantile_source = rq_unclipped

if hasattr(rq_unclipped.data, "rechunk"):
    spatial_axes = {
        rq_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }
    rq_quantile_source = rq_unclipped.copy(
        data=rq_unclipped.data.rechunk(spatial_axes)
    )

rq_daily_p95 = (
    rq_quantile_source
    .quantile(RQ_CLIP_PERCENTILE / 100.0, dim=SPATIAL_DIMS, skipna=True)
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)
rq_cube.name = "reliability_qualified_ntl"

print(
    "Daily P95 range:",
    f"{float(rq_daily_p95.min(skipna=True)):.2f}",
    "to",
    f"{float(rq_daily_p95.max(skipna=True)):.2f}",
    "nW cm⁻² sr⁻¹",
)

In [ ]:
# ============================================================
# 8. PROFILE FUNCTIONS
# ============================================================


def crop_to_support(cube, support_mask):
    support_values = np.asarray(support_mask.fillna(False).values, dtype=bool)
    rows, columns = np.where(support_values)

    if len(rows) == 0:
        return None, None

    y_slice = slice(rows.min(), rows.max() + 1)
    x_slice = slice(columns.min(), columns.max() + 1)

    return (
        cube.isel(y=y_slice, x=x_slice),
        support_mask.isel(y=y_slice, x=x_slice),
    )


def build_pixel_matched_profile(
    cube,
    support_mask,
    aggregation_days,
    unit_name,
    unit_type,
    method="Reliability-qualified DNB-BRDF",
):
    '''Build a daily or non-overlapping multi-day profile.'''

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(support_mask)
    )

    dates = pd.DatetimeIndex(selected["date"].values).normalize()
    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        aggregation_days,
    ).astype(int)

    composites = (
        selected
        .assign_coords(block=("date", block_numbers))
        .groupby("block")
        .mean(dim="date", skipna=True)
    )

    blocks = composites["block"].values.astype(int)
    block_start = EVENT_DATE + pd.to_timedelta(blocks * aggregation_days, unit="D")
    block_end = block_start + pd.Timedelta(days=aggregation_days - 1)

    baseline_blocks = blocks[
        (block_start >= BASELINE_START)
        & (block_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(f"{unit_name}: no complete baseline blocks were found.")

    baseline_composites = composites.sel(block=baseline_blocks).compute()
    baseline_observations = baseline_composites.notnull().sum(dim="block")

    ntl0 = baseline_composites.median(dim="block", skipna=True)
    baseline_quantiles = baseline_composites.quantile(
        [0.25, 0.75],
        dim="block",
        skipna=True,
    )
    ntl0_q25 = baseline_quantiles.sel(quantile=0.25, drop=True)
    ntl0_q75 = baseline_quantiles.sel(quantile=0.75, drop=True)

    minimum_baseline = MIN_BASELINE_OBSERVATIONS[aggregation_days]
    fixed_mask = (
        support_mask
        & (baseline_observations >= minimum_baseline)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(fixed_mask.sum().item())

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{unit_name}: no baseline-lit {SETTLEMENT_MASK} pixels "
            f"met the baseline requirement."
        )

    paired_valid = composites.notnull() & fixed_mask & ntl0.notnull()
    valid_pixel_count = paired_valid.sum(dim=SPATIAL_DIMS)
    spatial_coverage_pct = 100.0 * valid_pixel_count / fixed_pixel_count

    current_radiance = composites.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    matched_baseline = ntl0.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    recovery_pct = 100.0 * current_radiance / matched_baseline

    uncertainty_valid = (
        paired_valid
        & ntl0_q25.notnull()
        & ntl0_q75.notnull()
        & (ntl0_q25 > 0)
    )
    uncertainty_current = composites.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q25_sum = ntl0_q25.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q75_sum = ntl0_q75.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )

    recovery_low_pct = 100.0 * uncertainty_current / baseline_q75_sum
    recovery_high_pct = 100.0 * uncertainty_current / baseline_q25_sum

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": matched_baseline,
            "recovery_pct": recovery_pct,
            "recovery_low_pct": recovery_low_pct,
            "recovery_high_pct": recovery_high_pct,
            "spatial_coverage_pct": spatial_coverage_pct,
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(
        profile["block"] * aggregation_days,
        unit="D",
    )
    profile["date_end"] = profile["date_start"] + pd.Timedelta(
        days=aggregation_days - 1
    )
    profile["date_mid"] = profile["date_start"] + pd.to_timedelta(
        (aggregation_days - 1) / 2,
        unit="D",
    )

    below_gate = profile["spatial_coverage_pct"] < SPATIAL_COMPLETENESS_PCT
    profile.loc[
        below_gate,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "recovery_pct",
            "recovery_low_pct",
            "recovery_high_pct",
        ],
    ] = np.nan

    profile["mean_ntl"] = (
        profile["current_radiance"] / profile["valid_pixel_count"]
    )
    profile["observation_status"] = np.where(
        profile["recovery_pct"].notna(),
        "observed",
        "not observable",
    )
    profile["unit_name"] = unit_name
    profile["unit_type"] = unit_type
    profile["method"] = method
    profile["aggregation_days"] = aggregation_days
    profile["ghsl_mask"] = SETTLEMENT_MASK
    profile["sc_threshold_pct"] = SPATIAL_COMPLETENESS_PCT

    baseline_rows = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    ]

    report = {
        "unit_name": unit_name,
        "unit_type": unit_type,
        "aggregation_days": aggregation_days,
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_blocks": len(baseline_blocks),
        "minimum_baseline_observations": minimum_baseline,
        "fixed_baseline_pixels": fixed_pixel_count,
        "median_baseline_coverage_pct": baseline_rows[
            "spatial_coverage_pct"
        ].median(),
    }

    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

In [ ]:
# ============================================================
# 9. BUILD REGIONAL AND MUNICIPAL PROFILES
# ============================================================

regional_cube, regional_support = crop_to_support(rq_cube, rq_base_mask)

(
    regional_daily,
    regional_composites_daily,
    regional_ntl0_daily,
    regional_fixed_daily,
    regional_daily_report,
) = build_pixel_matched_profile(
    cube=regional_cube,
    support_mask=regional_support,
    aggregation_days=1,
    unit_name="Samar–Leyte MuniCities support",
    unit_type="Regional bridge",
)

(
    regional_four_day,
    regional_composites_four_day,
    regional_ntl0_four_day,
    regional_fixed_four_day,
    regional_four_day_report,
) = build_pixel_matched_profile(
    cube=regional_cube,
    support_mask=regional_support,
    aggregation_days=4,
    unit_name="Samar–Leyte MuniCities support",
    unit_type="Regional bridge",
)

municipal_profiles = []
municipal_reports = []
municipal_failures = []

for row in selected_raster_crs.itertuples():
    unit_support = (selected_zone_id == int(row.profile_id)) & ghsl_mask
    local_cube, local_support = crop_to_support(rq_cube, unit_support)

    if local_cube is None:
        municipal_failures.append(
            {
                "unit_name": row.unit_name,
                "reason": f"No {SETTLEMENT_MASK} pixels in the municipality",
            }
        )
        continue

    for aggregation_days in (1, 4):
        try:
            profile, _, _, _, report = build_pixel_matched_profile(
                cube=local_cube,
                support_mask=local_support,
                aggregation_days=aggregation_days,
                unit_name=row.unit_name,
                unit_type="City/municipality",
            )
            municipal_profiles.append(profile)
            municipal_reports.append(report)
        except ValueError as error:
            municipal_failures.append(
                {
                    "unit_name": row.unit_name,
                    "aggregation_days": aggregation_days,
                    "reason": str(error),
                }
            )

if not municipal_profiles:
    raise ValueError(
        f"No municipal profiles could be built with {SETTLEMENT_MASK}. "
        "Inspect municipal_failures or rerun with SETTLEMENT_MASK = 'G3'."
    )

municipal_profiles = pd.concat(municipal_profiles, ignore_index=True)
municipal_reports = pd.DataFrame(municipal_reports)
municipal_failures = pd.DataFrame(municipal_failures)

municipal_daily = municipal_profiles.loc[
    municipal_profiles["aggregation_days"] == 1
].copy()
municipal_four_day = municipal_profiles.loc[
    municipal_profiles["aggregation_days"] == 4
].copy()

if municipal_four_day.empty:
    raise ValueError(
        "Daily profiles were available, but no four-day municipal profile "
        "passed the baseline support requirement."
    )

print("Municipal profiles built:", municipal_four_day["unit_name"].nunique())
display(
    municipal_reports.loc[
        municipal_reports["aggregation_days"] == 4,
        [
            "unit_name",
            "fixed_baseline_pixels",
            "median_baseline_coverage_pct",
            "ghsl_mask",
            "sc_threshold_pct",
        ],
    ].sort_values("unit_name")
)

if not municipal_failures.empty:
    print("Profiles withheld or unavailable:")
    display(municipal_failures)

In [ ]:
# ============================================================
# 10. CHARACTERISTIC POI PROFILES
# ============================================================

poi_table = pd.DataFrame(
    [
        {
            "poi_name": "Tacloban City Centre",
            "group": "City centre",
            "longitude": 125.0015,
            "latitude": 11.2434,
            "rationale": "Regional capital; severe Haiyan impact; dense urban lighting",
        },
        {
            "poi_name": "Ormoc City Centre",
            "group": "City centre",
            "longitude": 124.6068,
            "latitude": 11.0088,
            "rationale": "Western Leyte urban hub and port",
        },
        {
            "poi_name": "Baybay City Centre",
            "group": "City centre",
            "longitude": 124.7989,
            "latitude": 10.6766,
            "rationale": "Smaller western coastal city with persistent urban lights",
        },
        {
            "poi_name": "Palo Centre",
            "group": "Municipal centre",
            "longitude": 124.9904,
            "latitude": 11.1577,
            "rationale": "Eastern Leyte urban corridor and major resettlement context",
        },
        {
            "poi_name": "Alangalang Centre",
            "group": "Municipal centre",
            "longitude": 124.8465,
            "latitude": 11.2071,
            "rationale": "Inland peri-urban exposure contrast",
        },
        {
            "poi_name": "Guiuan Centre",
            "group": "Municipal centre",
            "longitude": 125.7232,
            "latitude": 11.0312,
            "rationale": "Near Haiyan's first landfall",
        },
    ]
)


def poi_window(longitude, latitude, kernel_size):
    point = gpd.GeoSeries(
        [Point(longitude, latitude)],
        crs="EPSG:4326",
    ).to_crs(a2.rio.crs).iloc[0]

    x_index = int(np.abs(dnb["x"].values - point.x).argmin())
    y_index = int(np.abs(dnb["y"].values - point.y).argmin())
    half = kernel_size // 2

    x_slice = slice(max(0, x_index - half), min(dnb.sizes["x"], x_index + half + 1))
    y_slice = slice(max(0, y_index - half), min(dnb.sizes["y"], y_index + half + 1))

    local_cube = rq_cube.isel(x=x_slice, y=y_slice)
    local_support = (
        ghsl_mask.isel(x=x_slice, y=y_slice)
        & study_mask.isel(x=x_slice, y=y_slice)
    )

    return local_cube, local_support


poi_profiles = []
poi_reports = []
poi_failures = []

for poi in poi_table.itertuples():
    local_cube, local_support = poi_window(
        poi.longitude,
        poi.latitude,
        POI_KERNEL_SIZE,
    )

    for aggregation_days in (1, 4):
        try:
            profile, _, _, _, report = build_pixel_matched_profile(
                cube=local_cube,
                support_mask=local_support,
                aggregation_days=aggregation_days,
                unit_name=poi.poi_name,
                unit_type=f"POI {POI_KERNEL_SIZE}×{POI_KERNEL_SIZE}",
            )
            poi_profiles.append(profile)
            poi_reports.append(report)
        except ValueError as error:
            poi_failures.append(
                {
                    "poi_name": poi.poi_name,
                    "aggregation_days": aggregation_days,
                    "reason": str(error),
                }
            )

poi_profiles = pd.concat(poi_profiles, ignore_index=True) if poi_profiles else pd.DataFrame()
poi_reports = pd.DataFrame(poi_reports)
poi_failures = pd.DataFrame(poi_failures)

poi_daily = poi_profiles.loc[
    poi_profiles["aggregation_days"] == 1
].copy() if not poi_profiles.empty else pd.DataFrame()

poi_four_day = poi_profiles.loc[
    poi_profiles["aggregation_days"] == 4
].copy() if not poi_profiles.empty else pd.DataFrame()

In [ ]:
# ============================================================
# 11. NGCP REGIONAL CONSISTENCY CHECK
# ============================================================


def build_ngcp_profile(path):
    if not path.exists():
        raise FileNotFoundError(f"NGCP CSV not found:\n{path}")

    raw = pd.read_csv(path, skiprows=2, low_memory=False)
    raw.columns = [str(column).strip() for column in raw.columns]

    daily = pd.DataFrame(
        {
            "date": pd.to_datetime(
                raw["DATE"].astype(str).str.strip(),
                dayfirst=True,
                errors="coerce",
            ).dt.normalize(),
            "load_mw": pd.to_numeric(raw["1"], errors="coerce"),
        }
    ).dropna(subset=["date", "load_mw"])

    daily = (
        daily
        .groupby("date", as_index=False)["load_mw"]
        .mean()
        .sort_values("date")
    )
    daily = daily.loc[
        (daily["date"] >= ANALYSIS_START)
        & (daily["date"] <= PROFILE_END)
    ].copy()

    daily["block"] = np.floor_divide(
        (daily["date"] - EVENT_DATE).dt.days,
        4,
    ).astype(int)

    profile = daily.groupby("block", as_index=False)["load_mw"].mean()
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(profile["block"] * 4, unit="D")
    profile["date_end"] = profile["date_start"] + pd.Timedelta(days=3)

    baseline = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END),
        "load_mw",
    ].median()

    if not np.isfinite(baseline) or baseline <= 0:
        raise ValueError("NGCP baseline is unavailable or non-positive.")

    profile["recovery_pct"] = 100.0 * profile["load_mw"] / baseline
    return profile, baseline


def safe_correlation(x, y):
    paired = pd.DataFrame(
        {
            "x": pd.to_numeric(x, errors="coerce"),
            "y": pd.to_numeric(y, errors="coerce"),
        }
    ).dropna()

    if len(paired) < 3 or paired["x"].nunique() < 2 or paired["y"].nunique() < 2:
        return np.nan

    return paired["x"].corr(paired["y"])


ngcp_four_day, ngcp_baseline_mw = build_ngcp_profile(NGCP_CSV_PATH)

regional_ngcp = (
    regional_four_day[["date_start", "recovery_pct", "spatial_coverage_pct"]]
    .merge(
        ngcp_four_day[["date_start", "recovery_pct"]].rename(
            columns={"recovery_pct": "ngcp_recovery_pct"}
        ),
        on="date_start",
        how="inner",
    )
    .dropna(subset=["recovery_pct", "ngcp_recovery_pct"])
)

post_bridge = regional_ngcp.loc[regional_ngcp["date_start"] >= EVENT_DATE]
bridge_r = safe_correlation(post_bridge["recovery_pct"], post_bridge["ngcp_recovery_pct"])
bridge_n = len(post_bridge)

if bridge_n >= 20 and np.isfinite(bridge_r) and bridge_r >= 0.70:
    bridge_decision = (
        "Regional construct consistency is supported. Local trajectories may be analysed "
        "as conditional functional NTL profiles, but not labelled as municipal electricity restoration."
    )
elif bridge_n >= 12 and np.isfinite(bridge_r) and bridge_r >= 0.50:
    bridge_decision = (
        "Regional consistency is moderate. Local trajectories remain exploratory and require "
        "stronger external evidence."
    )
else:
    bridge_decision = (
        "The municipality-clipped regional bridge is weak or sparsely observed. Do not bank "
        "on NGCP alignment for local interpretation in this configuration."
    )

bridge_summary = pd.DataFrame(
    [
        {
            "GHSL mask": SETTLEMENT_MASK,
            "SC threshold (%)": SPATIAL_COMPLETENESS_PCT,
            "Paired post-event composites": bridge_n,
            "Pearson r": bridge_r,
            "NGCP baseline (MW)": ngcp_baseline_mw,
            "Decision": bridge_decision,
        }
    ]
)

In [ ]:
# ============================================================
# 12. METRIC FUNCTIONS
# ============================================================


def expected_event_blocks(aggregation_days=4):
    inclusive_days = (PROFILE_END - EVENT_DATE).days + 1
    return int(np.ceil(inclusive_days / aggregation_days))


def longest_missing_run_days(profile, aggregation_days=4):
    expected_blocks = np.arange(
        0,
        int(np.floor((PROFILE_END - EVENT_DATE).days / aggregation_days)) + 1,
    )
    observed_blocks = set(
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & profile["recovery_pct"].notna(),
            "block",
        ].astype(int)
    )
    missing = np.array([block not in observed_blocks for block in expected_blocks], dtype=int)

    longest = 0
    current = 0
    for value in missing:
        if value:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest * aggregation_days


def observability_class(profile, aggregation_days=4):
    post = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    retained = int(post["recovery_pct"].notna().sum())
    expected = expected_event_blocks(aggregation_days)
    retained_pct = 100.0 * retained / expected if expected else np.nan
    max_gap = longest_missing_run_days(profile, aggregation_days)

    if (
        retained >= MIN_EVENT_COMPOSITES
        and retained_pct >= MIN_EVENT_RETENTION_PCT
        and max_gap <= MAX_INTERPRETABLE_GAP_DAYS
    ):
        label = "interpretable with interval timing"
        flag = "OBS_OK"
    elif retained >= MIN_EVENT_COMPOSITES and retained_pct >= MIN_EVENT_RETENTION_PCT:
        label = "observation-limited"
        flag = "OBS_LIMITED"
    else:
        label = "not observable"
        flag = "NOT_OBSERVABLE"

    return {
        "retained_composites": retained,
        "expected_composites": expected,
        "retained_pct": retained_pct,
        "max_gap_days": max_gap,
        "observability": label,
        "quality_flag": flag,
    }


def persistent_crossing(profile, threshold, value_column="recovery_pct"):
    post = (
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & profile[value_column].notna()
        ]
        .sort_values("block")
        .set_index("block", drop=False)
    )

    for block, row in post.iterrows():
        required = list(range(int(block), int(block) + PERSISTENCE_BLOCKS))

        if not set(required).issubset(post.index):
            continue

        values = post.loc[required, value_column]

        if (values >= threshold).all():
            previous_below = post.loc[
                (post["block"] < block) & (post[value_column] < threshold)
            ]
            lower_day = 0

            if not previous_below.empty:
                lower_day = max(
                    0,
                    int(
                        (
                            previous_below.iloc[-1]["date_end"]
                            + pd.Timedelta(days=1)
                            - EVENT_DATE
                        ).days
                    ),
                )

            upper_day = int((row["date_start"] - EVENT_DATE).days)

            return {
                "day": upper_day,
                "lower_day": lower_day,
                "upper_day": upper_day,
                "date": row["date_start"],
            }

    return None


def format_interval(crossing):
    if crossing is None:
        return None
    return f"{crossing['lower_day']}–{crossing['upper_day']} d"


def format_baseline_range(optimistic, conservative):
    if optimistic is None and conservative is None:
        return None

    earliest = optimistic["day"] if optimistic is not None else None
    latest = conservative["day"] if conservative is not None else None

    if earliest is not None and latest is not None:
        return f"{min(earliest, latest)}–{max(earliest, latest)} d"
    if earliest is not None:
        return f"≥{earliest} d; conservative crossing absent"
    return f"≤{latest} d; optimistic crossing absent"


def calculate_recovery_metrics(profile):
    profile = profile.sort_values("date_start").copy()
    obs = observability_class(profile, aggregation_days=4)
    event = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    impact_window = event.loc[
        event["date_start"] <= EVENT_DATE + pd.Timedelta(days=59)
    ].dropna(subset=["recovery_pct"])

    if impact_window.empty:
        impact_recovery = np.nan
        impact_date = pd.NaT
        impact_drop = np.nan
        impact_drop_low = np.nan
        impact_drop_high = np.nan
    else:
        impact_row = impact_window.loc[impact_window["recovery_pct"].idxmin()]
        impact_recovery = float(impact_row["recovery_pct"])
        impact_date = impact_row["date_start"]
        impact_drop = 100.0 - impact_recovery
        impact_drop_low = 100.0 - float(impact_row["recovery_high_pct"])
        impact_drop_high = 100.0 - float(impact_row["recovery_low_pct"])

    crossings = {}
    for threshold in (50, 80, 90):
        central = persistent_crossing(profile, threshold, "recovery_pct")
        optimistic = persistent_crossing(profile, threshold, "recovery_high_pct")
        conservative = persistent_crossing(profile, threshold, "recovery_low_pct")

        if central is not None:
            status = "supported; interval-censored"
        elif obs["quality_flag"] == "NOT_OBSERVABLE":
            status = "not observable"
        elif obs["quality_flag"] == "OBS_LIMITED":
            status = "not identifiable: observation-limited"
        else:
            status = f"not recovered by {int((PROFILE_END - EVENT_DATE).days)} d"

        crossings[threshold] = {
            "central": central,
            "optimistic": optimistic,
            "conservative": conservative,
            "status": status,
        }

    post_observed = event.dropna(subset=["recovery_pct"]).copy()
    slope = np.nan

    if not post_observed.empty and pd.notna(impact_date):
        slope_end_day = (
            crossings[90]["central"]["day"]
            if crossings[90]["central"] is not None
            else int((PROFILE_END - EVENT_DATE).days)
        )
        slope_data = post_observed.loc[
            (post_observed["date_start"] >= impact_date)
            & (
                post_observed["date_start"]
                <= EVENT_DATE + pd.Timedelta(days=slope_end_day)
            )
        ]

        if len(slope_data) >= 3:
            x = (slope_data["date_start"] - EVENT_DATE).dt.days.to_numpy(dtype=float)
            y = slope_data["recovery_pct"].to_numpy(dtype=float)
            slope = float(np.polyfit(x, y, 1)[0])

    stage3 = event.loc[
        (event["date_start"] >= EVENT_DATE + pd.Timedelta(days=120))
        & (event["date_start"] <= EVENT_DATE + pd.Timedelta(days=179))
    ].dropna(subset=["recovery_pct"])

    stability_mad = (
        float(
            np.median(
                np.abs(stage3["recovery_pct"] - stage3["recovery_pct"].median())
            )
        )
        if not stage3.empty
        else np.nan
    )
    stable_share = (
        float(stage3["recovery_pct"].between(90, 110).mean() * 100.0)
        if not stage3.empty
        else np.nan
    )

    below_baseline_observed_days = int(
        (post_observed["recovery_pct"] < 100).sum() * 4
    )

    median_sc = float(event["spatial_coverage_pct"].median())
    minimum_sc = float(event["spatial_coverage_pct"].min())

    result = {
        "unit_name": profile["unit_name"].iloc[0],
        "unit_type": profile["unit_type"].iloc[0],
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "impact_date": impact_date,
        "impact_recovery_pct": impact_recovery,
        "impact_drop_pct": impact_drop,
        "impact_drop_range_pct": (
            f"{impact_drop_low:.1f}–{impact_drop_high:.1f}"
            if np.isfinite(impact_drop_low) and np.isfinite(impact_drop_high)
            else None
        ),
        "recovery_slope_pct_per_day": slope,
        "observed_days_below_baseline": below_baseline_observed_days,
        "stage3_stability_mad_pct": stability_mad,
        "stage3_within_90_110_pct": stable_share,
        "median_event_sc_pct": median_sc,
        "minimum_event_sc_pct": minimum_sc,
        **obs,
    }

    for threshold, crossing in crossings.items():
        result[f"T{threshold}_day"] = (
            crossing["central"]["day"]
            if crossing["central"] is not None
            else np.nan
        )
        result[f"T{threshold}_observation_interval"] = format_interval(
            crossing["central"]
        )
        result[f"T{threshold}_baseline_range"] = format_baseline_range(
            crossing["optimistic"],
            crossing["conservative"],
        )
        result[f"T{threshold}_status"] = crossing["status"]

    return result

In [ ]:
# ============================================================
# 13. MUNICIPALITY AND POI METRIC TABLES
# ============================================================

municipal_metrics = pd.DataFrame(
    [
        calculate_recovery_metrics(group)
        for _, group in municipal_four_day.groupby("unit_name", sort=True)
    ]
)

poi_metrics = (
    pd.DataFrame(
        [
            calculate_recovery_metrics(group)
            for _, group in poi_four_day.groupby("unit_name", sort=True)
        ]
    )
    if not poi_four_day.empty
    else pd.DataFrame()
)

In [ ]:
# ============================================================
# 14. STAGE SUMMARIES
# ============================================================


def summarise_stages(profile_table):
    rows = []

    for unit_name, profile in profile_table.groupby("unit_name", sort=True):
        for stage_name, (stage_start, stage_end) in STAGE_WINDOWS.items():
            stage = profile.loc[
                (profile["date_start"] >= stage_start)
                & (profile["date_end"] <= stage_end)
            ]
            inclusive_days = (stage_end - stage_start).days + 1
            expected = int(np.ceil(inclusive_days / 4))
            observed = stage.dropna(subset=["recovery_pct"])

            rows.append(
                {
                    "unit_name": unit_name,
                    "stage": stage_name,
                    "expected_composites": expected,
                    "observed_composites": len(observed),
                    "retained_pct": 100.0 * len(observed) / expected if expected else np.nan,
                    "median_recovery_pct": observed["recovery_pct"].median(),
                    "recovery_q25_pct": observed["recovery_pct"].quantile(0.25),
                    "recovery_q75_pct": observed["recovery_pct"].quantile(0.75),
                    "median_sc_pct": stage["spatial_coverage_pct"].median(),
                    "minimum_sc_pct": stage["spatial_coverage_pct"].min(),
                    "not_observable_composites": int(stage["recovery_pct"].isna().sum()),
                }
            )

    return pd.DataFrame(rows)


municipal_stage_summary = summarise_stages(municipal_four_day)
poi_stage_summary = (
    summarise_stages(poi_four_day)
    if not poi_four_day.empty
    else pd.DataFrame()
)

In [ ]:
# ============================================================
# 17. BUILD REGIONAL STAGE MAPS
# ============================================================


def stage_map_from_profile(stage_name, stage_start, stage_end):
    if stage_name == "Baseline":
        return regional_ntl0_four_day.compute()

    eligible = regional_four_day.loc[
        (regional_four_day["date_start"] >= stage_start)
        & (regional_four_day["date_end"] <= stage_end)
        & regional_four_day["recovery_pct"].notna()
    ]
    blocks = eligible["block"].astype(int).to_numpy()

    if len(blocks) == 0:
        return xr.full_like(
            regional_composites_four_day.isel(block=0),
            np.nan,
        ).compute()

    return (
        regional_composites_four_day
        .sel(block=blocks)
        .median(dim="block", skipna=True)
        .compute()
    )


stage_maps = {
    stage_name: stage_map_from_profile(stage_name, stage_start, stage_end)
    for stage_name, (stage_start, stage_end) in STAGE_WINDOWS.items()
}

map_values = []

for stage_map in stage_maps.values():
    values = np.asarray(stage_map.values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size:
        map_values.append(values)

if not map_values:
    raise ValueError("No stage-map radiance values were available.")

map_color_max = max(float(np.nanpercentile(np.concatenate(map_values), 98)), 1.0)


def extract_line_coordinates(geodataframe):
    x_values = []
    y_values = []

    def add_line(line):
        x, y = line.xy
        x_values.extend([*x, None])
        y_values.extend([*y, None])

    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue
        if geometry.geom_type == "LineString":
            add_line(geometry)
        elif geometry.geom_type == "MultiLineString":
            for line in geometry.geoms:
                add_line(line)
        elif geometry.geom_type == "Polygon":
            add_line(geometry.exterior)
        elif geometry.geom_type == "MultiPolygon":
            for polygon in geometry.geoms:
                add_line(polygon.exterior)

    return x_values, y_values


selected_boundaries = selected_raster_crs.copy()
selected_boundaries["geometry"] = selected_boundaries.geometry.boundary
boundary_x, boundary_y = extract_line_coordinates(selected_boundaries)

study_bounds = dnb.rio.bounds()
roads_for_map = roads_raster_crs.cx[
    study_bounds[0]:study_bounds[2],
    study_bounds[1]:study_bounds[3],
].copy()

if len(roads_for_map) > 12000:
    roads_for_map = roads_for_map.iloc[:: max(1, len(roads_for_map) // 12000)].copy()

road_x, road_y = extract_line_coordinates(roads_for_map)

## 1. From the regional signal to named locations

The selected LGUs span the severely affected Leyte Gulf and landfall corridor, an inland contrast, and western urban comparisons. POIs reproduce the RQ1 city-centre and municipal-centre supports. Western locations are comparisons, not unaffected controls.

In [ ]:
# ============================================================
# FIGURE 1. STUDY LOCATIONS
# ============================================================

selected_display = selected_municipalities.to_crs("EPSG:4326").copy()
roads_display = roads.to_crs("EPSG:4326")

overview_bounds = selected_display.total_bounds
x_margin = max((overview_bounds[2] - overview_bounds[0]) * 0.04, 0.01)
y_margin = max((overview_bounds[3] - overview_bounds[1]) * 0.04, 0.01)

roads_overview = roads_display.cx[
    overview_bounds[0] - x_margin:overview_bounds[2] + x_margin,
    overview_bounds[1] - y_margin:overview_bounds[3] + y_margin,
].copy()

if len(roads_overview) > 10000:
    roads_overview = roads_overview.iloc[
        ::max(1, len(roads_overview) // 10000)
    ].copy()

overview_road_x, overview_road_y = extract_line_coordinates(roads_overview)

overview_boundaries = selected_display.copy()
overview_boundaries["geometry"] = overview_boundaries.geometry.boundary
overview_boundary_x, overview_boundary_y = extract_line_coordinates(
    overview_boundaries
)

fig_locations = go.Figure()
fig_locations.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(color="#D6DCE4", width=1),
        name="Roads",
        hoverinfo="skip",
    )
)
fig_locations.add_trace(
    go.Scatter(
        x=overview_boundary_x,
        y=overview_boundary_y,
        mode="lines",
        line=dict(color="#526B82", width=2),
        name="Selected LGUs",
        hoverinfo="skip",
    )
)

centroids = selected_display.geometry.representative_point()
fig_locations.add_trace(
    go.Scatter(
        x=centroids.x,
        y=centroids.y,
        mode="markers+text",
        text=selected_display["unit_name"],
        textposition="top center",
        marker=dict(size=7, color="#243B5A", line=dict(color="white", width=0.8)),
        textfont=dict(size=10, color="#243B5A"),
        name="Cities and municipalities",
        hovertemplate="%{text}<extra></extra>",
    )
)
fig_locations.add_trace(
    go.Scatter(
        x=poi_table["longitude"],
        y=poi_table["latitude"],
        mode="markers+text",
        text=poi_table["poi_name"],
        textposition="bottom center",
        marker=dict(
            size=11,
            color="#FF5A36",
            symbol="diamond",
            line=dict(color="white", width=1.1),
        ),
        textfont=dict(size=10, color="#A2321C"),
        name="Characteristic POIs",
        hovertemplate="%{text}<extra></extra>",
    )
)

fig_locations.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1350,
    height=780,
    # title="Selected city, municipality, and POI supports",
    xaxis=dict(
        title="Longitude",
        range=[overview_bounds[0] - x_margin, overview_bounds[2] + x_margin],
        showgrid=False,
        zeroline=False,
    ),
    yaxis=dict(
        title="Latitude",
        range=[overview_bounds[1] - y_margin, overview_bounds[3] + y_margin],
        showgrid=False,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
    ),
    legend=dict(orientation="h", y=1.02, x=0),
    font=dict(family="Arial", size=13, color="#243B5A"),
    margin=dict(l=70, r=35, t=90, b=65),
)
fig_locations.show()

In [ ]:
# ============================================================
# GHSL G3 DISPLAY LAYER — OFFICIAL GHS-SMOD COLOURS
# ============================================================

G3_CLASSES = (22, 23, 30)

G3_CLASS_LABELS = {
    22: "Semi-dense urban cluster",
    23: "Dense urban cluster",
    30: "Urban centre",
}

# Official GHS-SMOD colours:
# 22 = #B37A00
# 23 = #7F3300
# 30 = #FF0000
#
# Positions are calculated for zmin=22 and zmax=30.
# Breaks occur halfway between 22/23 and 23/30.
G3_COLORSCALE = [
    [0.000000, "#B37A00"],
    [0.062499, "#B37A00"],
    [0.062500, "#7F3300"],
    [0.562499, "#7F3300"],
    [0.562500, "#FF0000"],
    [1.000000, "#FF0000"],
]

ghsl_g3_codes = (
    ghsl_viirs
    .where(ghsl_viirs.isin(G3_CLASSES))
    .rio.write_crs(ghsl_viirs.rio.crs)
)

ghsl_g3_display = (
    ghsl_g3_codes
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.nearest,
    )
    .rio.clip_box(
        minx=display_bounds[0],
        miny=display_bounds[1],
        maxx=display_bounds[2],
        maxy=display_bounds[3],
    )
)


# Use this trace as the background of fig_locations_g3.
fig_locations_g3 = go.Figure()

fig_locations_g3.add_trace(
    go.Heatmap(
        x=ghsl_g3_display["x"].values,
        y=ghsl_g3_display["y"].values,
        z=ghsl_g3_display.values,
        zmin=22,
        zmax=30,
        colorscale=G3_COLORSCALE,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            title="GHSL G3",
            tickmode="array",
            tickvals=[22, 23, 30],
            ticktext=[
                "22 — Semi-dense urban cluster",
                "23 — Dense urban cluster",
                "30 — Urban centre",
            ],
            thickness=17,
            len=0.55,
        ),
        hovertemplate=(
            "Longitude: %{x:.4f}<br>"
            "Latitude: %{y:.4f}<br>"
            "GHSL class: %{z:.0f}"
            "<extra></extra>"
        ),
        name="GHSL G3",
    )
)

# Retain your roads, LGU boundaries, centroids, POIs,
# axes, and layout traces below this point.


# ============================================================
# G3-MASKED RELIABILITY-QUALIFIED NTL BASELINE
# Fixes the Dask quantile chunking error
# ============================================================

ghsl_g3_mask = (
    ghsl_viirs
    .isin(G3_CLASSES)
    .fillna(False)
)

# Direct baseline observations only:
# DNB-BRDF, MQF == 0, G3 settlement pixels, municipal study area.
g3_baseline_unclipped = (
    dnb
    .sel(date=slice(BASELINE_START, PRE_EVENT_END))
    .where(
        (mqf.sel(date=slice(BASELINE_START, PRE_EVENT_END)) == 0)
        & ghsl_g3_mask
        & study_mask
    )
)

# Xarray's Dask quantile requires each reduced spatial dimension
# to be contained in a single chunk.
g3_quantile_source = g3_baseline_unclipped

if hasattr(g3_baseline_unclipped.data, "rechunk"):
    spatial_axes = {
        g3_baseline_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }

    g3_quantile_source = g3_baseline_unclipped.copy(
        data=g3_baseline_unclipped.data.rechunk(spatial_axes)
    )

# Daily G3-specific spatial P95.
g3_daily_p95 = (
    g3_quantile_source
    .quantile(
        RQ_CLIP_PERCENTILE / 100.0,
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    .squeeze(drop=True)
    .compute()
)

# Clamp values above the daily P95.
# This follows the principal RQ pipeline; it does not discard them.
g3_baseline_qualified = xr.where(
    g3_baseline_unclipped > g3_daily_p95,
    g3_daily_p95,
    g3_baseline_unclipped,
)

g3_baseline_count = g3_baseline_qualified.count("date")

g3_ntl0 = (
    g3_baseline_qualified
    .median("date", skipna=True)
    .where(
        g3_baseline_count
        >= MIN_BASELINE_OBSERVATIONS[1]
    )
    .compute()
)

g3_ntl0.name = "g3_reliability_qualified_baseline"

g3_ntl0_display = (
    g3_ntl0
    .rio.write_crs(a2.rio.crs)
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.bilinear,
    )
    .rio.clip_box(
        minx=display_bounds[0],
        miny=display_bounds[1],
        maxx=display_bounds[2],
        maxy=display_bounds[3],
    )
)

g3_baseline_values = np.asarray(
    g3_ntl0_display.values,
    dtype=float,
)

g3_baseline_values = g3_baseline_values[
    np.isfinite(g3_baseline_values)
]

if not g3_baseline_values.size:
    raise ValueError(
        "No reliability-qualified G3 baseline pixels were available."
    )

g3_baseline_color_max = max(
    float(
        np.nanpercentile(
            g3_baseline_values,
            98,
        )
    ),
    1.0,
)


In [ ]:


# ============================================================
# G3-MASKED BASELINE FIGURE
# ============================================================

fig_locations_g3_ntl = go.Figure()

fig_locations_g3_ntl.add_trace(
    go.Heatmap(
        x=g3_ntl0_display["x"].values,
        y=g3_ntl0_display["y"].values,
        z=g3_ntl0_display.values,
        colorscale="Inferno",
        zmin=0,
        zmax=3,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            title="Baseline NTL<br>(nW cm⁻² sr⁻¹)",
            thickness=17,
            len=0.55,
        ),
        hovertemplate=(
            "Longitude: %{x:.4f}<br>"
            "Latitude: %{y:.4f}<br>"
            "DNB-BRDF: %{z:.2f} nW cm⁻² sr⁻¹"
            "<extra></extra>"
        ),
        name="G3-masked baseline NTL",
    )
)

# Add the existing road layer.
fig_locations_g3_ntl.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="rgba(255,255,255,0.72)",
            width=0.1,
        ),
        name="Roads",
        hoverinfo="skip",
    )
)

# Add the selected LGU boundaries.
fig_locations_g3_ntl.add_trace(
    go.Scatter(
        x=overview_boundary_x,
        y=overview_boundary_y,
        mode="lines",
        line=dict(
            color="#56CCF2",
            width=0.5,
        ),
        name="Selected LGUs",
        hoverinfo="skip",
    )
)

# # Add city and municipality labels.
# fig_locations_g3_ntl.add_trace(
#     go.Scatter(
#         x=centroids.x,
#         y=centroids.y,
#         mode="markers+text",
#         text=selected_display["unit_name"],
#         textposition="top center",
#         marker=dict(
#             size=7,
#             color="#EAF7FF",
#             line=dict(
#                 color="#243B5A",
#                 width=1,
#             ),
#         ),
#         textfont=dict(
#             size=10,
#             color="#EAF7FF",
#         ),
#         name="Cities and municipalities",
#         hovertemplate="%{text}<extra></extra>",
#     )
# )

# # Add characteristic POIs.
# fig_locations_g3_ntl.add_trace(
#     go.Scatter(
#         x=poi_table["longitude"],
#         y=poi_table["latitude"],
#         mode="markers+text",
#         text=poi_table["poi_name"],
#         textposition="bottom center",
#         marker=dict(
#             size=11,
#             color="#00E5FF",
#             symbol="diamond",
#             line=dict(
#                 color="#17202A",
#                 width=1.1,
#             ),
#         ),
#         textfont=dict(
#             size=10,
#             color="#00E5FF",
#         ),
#         name="Characteristic POIs",
#         hovertemplate="%{text}<extra></extra>",
#     )
# )

fig_locations_g3_ntl.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#000000",
    width=1350,
    height=780,
    xaxis=dict(
        title="Longitude",
        range=[
            display_bounds[0],
            display_bounds[2],
        ],
        showgrid=False,
        zeroline=False,
    ),
    yaxis=dict(
        title="Latitude",
        range=[
            display_bounds[1],
            display_bounds[3],
        ],
        showgrid=False,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
    ),
    legend=dict(
        orientation="h",
        y=1.05,
        x=0,
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=70,
        r=170,
        t=90,
        b=65,
    ),
)

fig_locations_g3_ntl.show()

## 2. Regional baseline and stage recovery

The first panel shows baseline DNB-BRDF radiance. Post-event panels show median pixel recovery relative to that baseline. Roads and selected LGU boundaries provide spatial context; grey areas are outside the fixed reliability-qualified settlement support or were not observed sufficiently for that stage.

In [ ]:
# ============================================================
# FIGURE 2. REGIONAL BASELINE AND RECOVERY MAPS
# ============================================================

regional_recovery_maps = {
    stage_name: xr.where(
        regional_ntl0_four_day > 0,
        100.0 * stage_map / regional_ntl0_four_day,
        np.nan,
    )
    for stage_name, stage_map in stage_maps.items()
    if stage_name != "Baseline"
}

stage_names = list(STAGE_WINDOWS.keys())

fig_regional_recovery = make_subplots(
    rows=1,
    cols=len(stage_names),
    horizontal_spacing=0.012,
    subplot_titles=stage_names,
)

for column, stage_name in enumerate(stage_names, start=1):
    if stage_name == "Baseline":
        map_data = stage_maps[stage_name]
        fig_regional_recovery.add_trace(
            go.Heatmap(
                x=map_data["x"].values,
                y=map_data["y"].values,
                z=map_data.values,
                colorscale="Inferno",
                zmin=0,
                zmax=map_color_max,
                showscale=False,
                zsmooth=False,
                hoverongaps=False,
                hovertemplate="Baseline: %{z:.2f} nW cm⁻² sr⁻¹<extra></extra>",
            ),
            row=1,
            col=column,
        )
    else:
        map_data = regional_recovery_maps[stage_name]
        fig_regional_recovery.add_trace(
            go.Heatmap(
                x=map_data["x"].values,
                y=map_data["y"].values,
                z=map_data.values,
                coloraxis="coloraxis",
                zsmooth=False,
                hoverongaps=False,
                hovertemplate="Recovery: %{z:.1f}%<extra></extra>",
            ),
            row=1,
            col=column,
        )

    fig_regional_recovery.add_trace(
        go.Scatter(
            x=road_x,
            y=road_y,
            mode="lines",
            line=dict(color="rgba(255,255,255,0.50)", width=0.1),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
    fig_regional_recovery.add_trace(
        go.Scatter(
            x=boundary_x,
            y=boundary_y,
            mode="lines",
            line=dict(color="#D9F0FF", width=0.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
    fig_regional_recovery.update_xaxes(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=1,
        col=column,
    )
    fig_regional_recovery.update_yaxes(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=1,
        col=column,
    )

fig_regional_recovery.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#2E2E2E",
    width=2200,
    height=610,
    title=(
        f"Baseline nighttime lights and stage recovery | {SETTLEMENT_MASK} | "
        f"SC ≥ {SPATIAL_COMPLETENESS_PCT:.0f}%"
    ),
    coloraxis=dict(
        colorscale=[
            [0.00, "#8E1B1B"],
            [0.35, "#E67E22"],
            [0.70, "#F4D03F"],
            [100 / 140, "#F5F5F5"],
            [0.86, "#00F7FF"],
            [1.00, "#4800FF"],
        ],
        cmin=0,
        cmax=140,
        colorbar=dict(
            title="Recovery<br>(% baseline)",
            tickvals=[0, 50, 80, 100, 120, 140],
            thickness=18,
        ),
    ),
    font=dict(family="Arial", size=12, color="#243B5A"),
    margin=dict(l=30, r=120, t=110, b=35),
)
fig_regional_recovery.show()

In [ ]:
# ============================================================
# FIGURE 2. LOCATION-ZOOMED BASELINE AND RECOVERY MAPS
# ============================================================

# Change this to any valid municipality/city name in
# municipalities_raster_crs["unit_name"].
ZOOM_LOCATION = "Tacloban City"

# Increase for more surrounding context; decrease for a tighter crop.
ZOOM_PADDING_FRACTION = 0.15
ZOOM_MIN_PADDING_PIXELS = 4

# Show characteristic POIs falling inside the zoom extent.
SHOW_LOCAL_POIS = True


def subset_zoom_map(data_array, bounds):
    x_min, y_min, x_max, y_max = bounds

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(x_min, x_max)
        if x_values[0] < x_values[-1]
        else slice(x_max, x_min)
    )

    y_slice = (
        slice(y_min, y_max)
        if y_values[0] < y_values[-1]
        else slice(y_max, y_min)
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


# ------------------------------------------------------------
# FIND THE REQUESTED CITY OR MUNICIPALITY
# ------------------------------------------------------------

zoom_key = canonical_unit_name(
    ZOOM_LOCATION
)

zoom_match = municipalities_raster_crs.loc[
    municipalities_raster_crs["unit_key"] == zoom_key
].copy()

if zoom_match.empty:
    available_matches = (
        municipalities_raster_crs.loc[
            municipalities_raster_crs[
                "unit_name"
            ].str.contains(
                ZOOM_LOCATION,
                case=False,
                regex=False,
                na=False,
            ),
            "unit_name",
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    raise ValueError(
        f"No exact municipality match for '{ZOOM_LOCATION}'. "
        f"Possible matches: {available_matches}"
    )

if len(zoom_match) > 1:
    print(
        f"Multiple matches found for {ZOOM_LOCATION}; "
        "using the first feature."
    )

zoom_unit = zoom_match.iloc[[0]].copy()
zoom_unit_name = zoom_unit["unit_name"].iloc[0]

unit_x_min, unit_y_min, unit_x_max, unit_y_max = (
    zoom_unit.total_bounds
)

unit_width = unit_x_max - unit_x_min
unit_height = unit_y_max - unit_y_min

x_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["x"].values)
        )
    )
)

y_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["y"].values)
        )
    )
)

x_padding = max(
    unit_width * ZOOM_PADDING_FRACTION,
    x_resolution * ZOOM_MIN_PADDING_PIXELS,
)

y_padding = max(
    unit_height * ZOOM_PADDING_FRACTION,
    y_resolution * ZOOM_MIN_PADDING_PIXELS,
)

zoom_bounds = (
    unit_x_min - x_padding,
    unit_y_min - y_padding,
    unit_x_max + x_padding,
    unit_y_max + y_padding,
)


# ------------------------------------------------------------
# BUILD LOCAL ROAD AND MUNICIPAL BOUNDARY CONTEXT
# ------------------------------------------------------------

zoom_roads = roads_raster_crs.cx[
    zoom_bounds[0]:zoom_bounds[2],
    zoom_bounds[1]:zoom_bounds[3],
].copy()

if len(zoom_roads) > 6000:
    zoom_roads = zoom_roads.iloc[
        ::max(
            1,
            len(zoom_roads) // 6000,
        )
    ].copy()

zoom_road_x, zoom_road_y = extract_line_coordinates(
    zoom_roads
)

zoom_context_units = municipalities_raster_crs.cx[
    zoom_bounds[0]:zoom_bounds[2],
    zoom_bounds[1]:zoom_bounds[3],
].copy()

zoom_context_units["geometry"] = (
    zoom_context_units.geometry.boundary
)

zoom_boundary_x, zoom_boundary_y = extract_line_coordinates(
    zoom_context_units
)

zoom_target_boundary = zoom_unit.copy()
zoom_target_boundary["geometry"] = (
    zoom_target_boundary.geometry.boundary
)

zoom_target_x, zoom_target_y = extract_line_coordinates(
    zoom_target_boundary
)


# ------------------------------------------------------------
# PREPARE LOCAL POIS
# ------------------------------------------------------------

poi_points_raster_crs = gpd.GeoDataFrame(
    poi_table.copy(),
    geometry=gpd.points_from_xy(
        poi_table["longitude"],
        poi_table["latitude"],
    ),
    crs="EPSG:4326",
).to_crs(a2.rio.crs)

zoom_pois = poi_points_raster_crs.cx[
    zoom_bounds[0]:zoom_bounds[2],
    zoom_bounds[1]:zoom_bounds[3],
].copy()


# ------------------------------------------------------------
# BUILD RECOVERY MAPS
# ------------------------------------------------------------

regional_recovery_maps = {
    stage_name: xr.where(
        regional_ntl0_four_day > 0,
        100.0
        * stage_map
        / regional_ntl0_four_day,
        np.nan,
    )
    for stage_name, stage_map in stage_maps.items()
    if stage_name != "Baseline"
}

stage_names = list(
    STAGE_WINDOWS.keys()
)

fig_zoom_recovery = make_subplots(
    rows=1,
    cols=len(stage_names),
    horizontal_spacing=0.012,
    subplot_titles=stage_names,
)

for column, stage_name in enumerate(
    stage_names,
    start=1,
):

    # --------------------------------------------------------
    # BASELINE OR RECOVERY RASTER
    # --------------------------------------------------------

    if stage_name == "Baseline":

        map_data = subset_zoom_map(
            stage_maps[stage_name],
            zoom_bounds,
        )

        fig_zoom_recovery.add_trace(
            go.Heatmap(
                x=map_data["x"].values,
                y=map_data["y"].values,
                z=map_data.values,
                colorscale="Inferno",
                zmin=0,
                zmax=map_color_max,
                showscale=False,
                zsmooth=False,
                hoverongaps=False,
                hovertemplate=(
                    "Baseline: %{z:.2f} "
                    "nW cm⁻² sr⁻¹"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )

    else:

        map_data = subset_zoom_map(
            regional_recovery_maps[
                stage_name
            ],
            zoom_bounds,
        )

        fig_zoom_recovery.add_trace(
            go.Heatmap(
                x=map_data["x"].values,
                y=map_data["y"].values,
                z=map_data.values,
                coloraxis="coloraxis",
                zsmooth=False,
                hoverongaps=False,
                hovertemplate=(
                    "Recovery: %{z:.1f}%"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )

    # --------------------------------------------------------
    # LOCAL ROADS
    # --------------------------------------------------------

    fig_zoom_recovery.add_trace(
        go.Scatter(
            x=zoom_road_x,
            y=zoom_road_y,
            mode="lines",
            line=dict(
                color="rgba(255,255,255,0.65)",
                width=0.8,
            ),
            hoverinfo="skip",
            showlegend=(
                column == 1
            ),
            name="Roads",
        ),
        row=1,
        col=column,
    )

    # --------------------------------------------------------
    # SURROUNDING MUNICIPAL BOUNDARIES
    # --------------------------------------------------------

    fig_zoom_recovery.add_trace(
        go.Scatter(
            x=zoom_boundary_x,
            y=zoom_boundary_y,
            mode="lines",
            line=dict(
                color="rgba(217,240,255,0.65)",
                width=0.5,
            ),
            hoverinfo="skip",
            showlegend=(
                column == 1
            ),
            name="Municipal boundaries",
        ),
        row=1,
        col=column,
    )

    # --------------------------------------------------------
    # TARGET MUNICIPALITY BOUNDARY
    # --------------------------------------------------------

    fig_zoom_recovery.add_trace(
        go.Scatter(
            x=zoom_target_x,
            y=zoom_target_y,
            mode="lines",
            line=dict(
                color="#00E5FF",
                width=0.25,
            ),
            hoverinfo="skip",
            showlegend=(
                column == 1
            ),
            name=zoom_unit_name,
        ),
        row=1,
        col=column,
    )

    # --------------------------------------------------------
    # LOCAL CHARACTERISTIC POIS
    # --------------------------------------------------------

    if SHOW_LOCAL_POIS and not zoom_pois.empty:

        fig_zoom_recovery.add_trace(
            go.Scatter(
                x=zoom_pois.geometry.x,
                y=zoom_pois.geometry.y,
                mode="markers",
                text=zoom_pois["poi_name"],
                marker=dict(
                    size=10,
                    color="#FF3B30",
                    symbol="diamond",
                    line=dict(
                        color="white",
                        width=1.2,
                    ),
                ),
                hovertemplate=(
                    "%{text}"
                    "<extra></extra>"
                ),
                showlegend=(
                    column == 1
                ),
                name="Characteristic POI",
            ),
            row=1,
            col=column,
        )

    # --------------------------------------------------------
    # IDENTICAL MAP EXTENT FOR EVERY STAGE
    # --------------------------------------------------------

    fig_zoom_recovery.update_xaxes(
        range=[
            zoom_bounds[0],
            zoom_bounds[2],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        constrain="domain",
        row=1,
        col=column,
    )

    fig_zoom_recovery.update_yaxes(
        range=[
            zoom_bounds[1],
            zoom_bounds[3],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        constrain="domain",
        row=1,
        col=column,
    )


# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig_zoom_recovery.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#2E2E2E",
    width=1600,
    height=400,

    title=(
        f"{zoom_unit_name}: baseline nighttime lights "
        "and stage recovery"
        f" | {SETTLEMENT_MASK}"
        f" | SC ≥ "
        f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
    ),

    coloraxis=dict(
        colorscale=[
            [0.00, "#8E1B1B"],
            [0.35, "#E67E22"],
            [0.70, "#F4D03F"],
            [100 / 140, "#F5F5F5"],
            [0.86, "#00F7FF"],
            [1.00, "#4800FF"],
        ],
        cmin=0,
        cmax=140,
        colorbar=dict(
            title=(
                "Recovery<br>"
                "(% baseline)"
            ),
            tickvals=[
                0,
                50,
                80,
                100,
                120,
                140,
            ],
            thickness=18,
        ),
    ),

    legend=dict(
        orientation="h",
        x=0.6,
        y=1.2,
    ),

    font=dict(
        family="Arial",
        size=12,
        color="#243B5A",
    ),

    margin=dict(
        l=30,
        r=120,
        t=125,
        b=35,
    ),
)

fig_zoom_recovery.update_annotations(
    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    )
)

fig_zoom_recovery.show()

## 3. Zoom in: maps, observability, and trajectory at each POI

Each location is read as one unit. The top row shows baseline radiance followed by stage recovery, with roads, administrative boundaries, the POI centre, and the 5×5 analytical kernel. The middle strip is spatial completeness. The lower panel shows raw daily recovery and the four-day profile. Error bars represent baseline sensitivity from the pixel-level baseline interquartile range.

In [ ]:
# ============================================================
# FIGURES 3A–3F. POI STORY PANELS
# ============================================================

POI_MAP_RADIUS_PIXELS = 9


def subset_bbox(data_array, bounds):
    x_min, y_min, x_max, y_max = bounds
    x_values = data_array["x"].values
    y_values = data_array["y"].values
    x_slice = slice(x_min, x_max) if x_values[0] < x_values[-1] else slice(x_max, x_min)
    y_slice = slice(y_min, y_max) if y_values[0] < y_values[-1] else slice(y_max, y_min)
    return data_array.sel(x=x_slice, y=y_slice)


def poi_in_raster_crs(longitude, latitude):
    return gpd.GeoSeries(
        [Point(longitude, latitude)],
        crs="EPSG:4326",
    ).to_crs(a2.rio.crs).iloc[0]


def rectangle_line(bounds):
    x_min, y_min, x_max, y_max = bounds
    return (
        [x_min, x_max, x_max, x_min, x_min],
        [y_min, y_min, y_max, y_max, y_min],
    )


x_resolution = float(np.median(np.abs(np.diff(dnb["x"].values))))
y_resolution = float(np.median(np.abs(np.diff(dnb["y"].values))))
poi_story_figures = {}

for poi in poi_table.itertuples():
    if poi_four_day.empty or poi.poi_name not in set(poi_four_day["unit_name"]):
        print(f"{poi.poi_name}: no admissible POI profile; map/time-series panel withheld.")
        continue

    point = poi_in_raster_crs(poi.longitude, poi.latitude)
    display_bounds = (
        point.x - POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y - POI_MAP_RADIUS_PIXELS * y_resolution,
        point.x + POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y + POI_MAP_RADIUS_PIXELS * y_resolution,
    )
    kernel_half = POI_KERNEL_SIZE / 2
    kernel_bounds = (
        point.x - kernel_half * x_resolution,
        point.y - kernel_half * y_resolution,
        point.x + kernel_half * x_resolution,
        point.y + kernel_half * y_resolution,
    )
    kernel_x, kernel_y = rectangle_line(kernel_bounds)

    local_roads = roads_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()
    local_road_x, local_road_y = extract_line_coordinates(local_roads)

    local_units = municipalities_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()
    local_units["geometry"] = local_units.geometry.boundary
    local_boundary_x, local_boundary_y = extract_line_coordinates(local_units)

    daily = poi_daily.loc[poi_daily["unit_name"] == poi.poi_name].sort_values("date_start")
    four_day = poi_four_day.loc[
        poi_four_day["unit_name"] == poi.poi_name
    ].sort_values("date_start")

    figure = make_subplots(
        rows=3,
        cols=len(stage_names),
        specs=[
            [{} for _ in stage_names],
            [{"colspan": len(stage_names)}] + [None] * (len(stage_names) - 1),
            [{"colspan": len(stage_names)}] + [None] * (len(stage_names) - 1),
        ],
        row_heights=[0.54, 0.14, 0.32],
        horizontal_spacing=0.012,
        vertical_spacing=0.075,
        subplot_titles=stage_names,
    )

    for column, stage_name in enumerate(stage_names, start=1):
        if stage_name == "Baseline":
            local_map = subset_bbox(regional_ntl0_four_day, display_bounds)
            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    colorscale="Inferno",
                    zmin=0,
                    zmax=map_color_max,
                    showscale=False,
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate="Baseline: %{z:.2f} nW cm⁻² sr⁻¹<extra></extra>",
                ),
                row=1,
                col=column,
            )
        else:
            local_map = subset_bbox(regional_recovery_maps[stage_name], display_bounds)
            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate="Recovery: %{z:.1f}%<extra></extra>",
                ),
                row=1,
                col=column,
            )

        figure.add_trace(
            go.Scatter(
                x=local_road_x,
                y=local_road_y,
                mode="lines",
                line=dict(color="rgba(255,255,255,0.72)", width=0.9),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )
        figure.add_trace(
            go.Scatter(
                x=local_boundary_x,
                y=local_boundary_y,
                mode="lines",
                line=dict(color="#56CCF2", width=1.1),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )
        figure.add_trace(
            go.Scatter(
                x=kernel_x,
                y=kernel_y,
                mode="lines",
                line=dict(color="#00E5FF", width=1.8, dash="dash"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )
        figure.add_trace(
            go.Scatter(
                x=[point.x],
                y=[point.y],
                mode="markers",
                marker=dict(
                    size=8,
                    color="#FF3B30",
                    symbol="x",
                    line=dict(width=1.2),
                ),
                hovertemplate=f"{poi.poi_name}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=column,
        )
        figure.update_xaxes(
            range=[display_bounds[0], display_bounds[2]],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )
        figure.update_yaxes(
            range=[display_bounds[1], display_bounds[3]],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

    figure.add_trace(
        go.Bar(
            x=four_day["date_start"],
            y=four_day["spatial_coverage_pct"],
            name="Spatial completeness",
            marker_color="#91AEC4",
            opacity=0.78,
            hovertemplate="%{x|%d %b %Y}<br>SC: %{y:.1f}%<extra></extra>",
        ),
        row=2,
        col=1,
    )
    figure.add_trace(
        go.Scatter(
            x=daily["date_start"],
            y=daily["recovery_pct"],
            mode="lines+markers",
            name="Daily RQ NTL",
            connectgaps=False,
            line=dict(color="#63BE7B", width=1.0),
            marker=dict(size=2.7),
            opacity=0.55,
            hovertemplate="%{x|%d %b %Y}<br>Daily: %{y:.1f}%<extra></extra>",
        ),
        row=3,
        col=1,
    )

    error_plus = (four_day["recovery_high_pct"] - four_day["recovery_pct"]).clip(lower=0)
    error_minus = (four_day["recovery_pct"] - four_day["recovery_low_pct"]).clip(lower=0)

    figure.add_trace(
        go.Scatter(
            x=four_day["date_start"],
            y=four_day["recovery_pct"],
            mode="lines+markers",
            name="RQ NTL (4D) ± baseline IQR",
            connectgaps=False,
            line=dict(color="#008F3D", width=2.7, shape="hv"),
            marker=dict(size=5),
            error_y=dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color="#006C2E",
                thickness=1.1,
                width=2,
            ),
            hovertemplate="%{x|%d %b %Y}<br>Recovery: %{y:.1f}%<extra></extra>",
        ),
        row=3,
        col=1,
    )

    figure.add_hline(
        y=SPATIAL_COMPLETENESS_PCT,
        line=dict(color="#C0392B", width=1.2, dash="dot"),
        row=2,
        col=1,
    )
    figure.add_hline(
        y=100,
        line=dict(color="#7F8C8D", width=1.2, dash="dot"),
        row=3,
        col=1,
    )
    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(color="#0057FF", width=2.0, dash="dash"),
        row=2,
        col=1,
    )
    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(color="#0057FF", width=2.0, dash="dash"),
        row=3,
        col=1,
    )

    figure.update_yaxes(
        title_text="SC (%)",
        range=[0, 105],
        showgrid=True,
        gridcolor="#E8EDF3",
        row=2,
        col=1,
    )
    figure.update_yaxes(
        title_text="Recovery<br>(% baseline)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )
    figure.update_xaxes(showticklabels=False, row=2, col=1)
    figure.update_xaxes(title_text="Date", row=3, col=1)

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1600,
        height=900,
        title=dict(
            text=(
                f"{poi.poi_name} | {poi.rationale}<br>"
                f"<sup>{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} VIIRS support; "
                f"{SETTLEMENT_MASK}; SC ≥ {SPATIAL_COMPLETENESS_PCT:.0f}%</sup>"
            ),
            x=0.5,
            xanchor="center",
        ),
        coloraxis=dict(
        colorscale=[
            [0.00, "#8E1B1B"],
            [0.35, "#E67E22"],
            [0.70, "#F4D03F"],
            [100 / 140, "#F5F5F5"],
            [0.86, "#00F7FF"],
            [1.00, "#4800FF"],
        ],
            cmin=0,
            cmax=140,
            colorbar=dict(
                title="Recovery<br>(% baseline)",
                tickvals=[0, 50, 80, 100, 120, 140],
                x=1.005,
                y=0.79,
                len=0.40,
                thickness=16,
            ),
        ),
        legend=dict(orientation="h", y=0.3, x=0.5),
        font=dict(family="Arial", size=16, color="#243B5A"),
        margin=dict(l=90, r=125, t=125, b=65),
        hovermode="x unified",
    )

    poi_story_figures[poi.poi_name] = figure
    figure.show()

In [ ]:
# ============================================================
# FIGURES 3A–3F. POI STORY PANELS
# ============================================================

POI_MAP_RADIUS_PIXELS = 9

SC_GREEN_COLORSCALE = [
    [0.00, "rgba(255,255,255,0.00)"],
    [0.20, "rgba(220,242,215,0.25)"],
    [0.40, "rgba(166,219,160,0.45)"],
    [0.60, "rgba(90,174,110,0.65)"],
    [0.80, "rgba(25,130,70,0.82)"],
    [1.00, "rgba(0,88,43,0.98)"],
]


def subset_bbox(data_array, bounds):
    x_min, y_min, x_max, y_max = bounds

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(x_min, x_max)
        if x_values[0] < x_values[-1]
        else slice(x_max, x_min)
    )

    y_slice = (
        slice(y_min, y_max)
        if y_values[0] < y_values[-1]
        else slice(y_max, y_min)
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


def poi_in_raster_crs(longitude, latitude):
    return (
        gpd.GeoSeries(
            [Point(longitude, latitude)],
            crs="EPSG:4326",
        )
        .to_crs(a2.rio.crs)
        .iloc[0]
    )


def rectangle_line(bounds):
    x_min, y_min, x_max, y_max = bounds

    return (
        [x_min, x_max, x_max, x_min, x_min],
        [y_min, y_min, y_max, y_max, y_min],
    )


x_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["x"].values)
        )
    )
)

y_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["y"].values)
        )
    )
)

poi_story_figures = {}

for poi in poi_table.itertuples():

    if (
        poi_four_day.empty
        or poi.poi_name not in set(poi_four_day["unit_name"])
    ):
        print(
            f"{poi.poi_name}: no admissible POI profile; "
            "map/time-series panel withheld."
        )
        continue

    point = poi_in_raster_crs(
        poi.longitude,
        poi.latitude,
    )

    display_bounds = (
        point.x - POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y - POI_MAP_RADIUS_PIXELS * y_resolution,
        point.x + POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y + POI_MAP_RADIUS_PIXELS * y_resolution,
    )

    kernel_half = POI_KERNEL_SIZE / 2

    kernel_bounds = (
        point.x - kernel_half * x_resolution,
        point.y - kernel_half * y_resolution,
        point.x + kernel_half * x_resolution,
        point.y + kernel_half * y_resolution,
    )

    kernel_x, kernel_y = rectangle_line(
        kernel_bounds
    )

    local_roads = roads_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()

    local_road_x, local_road_y = extract_line_coordinates(
        local_roads
    )

    local_units = municipalities_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()

    local_units["geometry"] = local_units.geometry.boundary

    local_boundary_x, local_boundary_y = extract_line_coordinates(
        local_units
    )

    daily = (
        poi_daily.loc[
            poi_daily["unit_name"] == poi.poi_name
        ]
        .sort_values("date_start")
        .copy()
    )

    four_day = (
        poi_four_day.loc[
            poi_four_day["unit_name"] == poi.poi_name
        ]
        .sort_values("date_start")
        .copy()
    )

    figure = make_subplots(
        rows=3,
        cols=len(stage_names),
        specs=[
            [{} for _ in stage_names],
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
        ],
        row_heights=[
            0.5,
            0.1,
            0.4,
        ],
        horizontal_spacing=0.012,
        vertical_spacing=0.075,
        subplot_titles=stage_names,
    )

    # --------------------------------------------------------
    # ROW 1. BASELINE AND RECOVERY MAPS
    # --------------------------------------------------------

    for column, stage_name in enumerate(
        stage_names,
        start=1,
    ):

        if stage_name == "Baseline":

            local_map = subset_bbox(
                regional_ntl0_four_day,
                display_bounds,
            )

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    colorscale="Inferno",
                    zmin=0,
                    zmax=map_color_max,
                    showscale=False,
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "Baseline: %{z:.2f} "
                        "nW cm⁻² sr⁻¹"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        else:

            local_map = subset_bbox(
                regional_recovery_maps[stage_name],
                display_bounds,
            )

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "Recovery: %{z:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        figure.add_trace(
            go.Scatter(
                x=local_road_x,
                y=local_road_y,
                mode="lines",
                line=dict(
                    color="rgba(255,255,255,0.72)",
                    width=0.9,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=local_boundary_x,
                y=local_boundary_y,
                mode="lines",
                line=dict(
                    color="#56CCF2",
                    width=1.1,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=kernel_x,
                y=kernel_y,
                mode="lines",
                line=dict(
                    color="#00E5FF",
                    width=1.8,
                    dash="dash",
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=[point.x],
                y=[point.y],
                mode="markers",
                marker=dict(
                    size=8,
                    color="#FF3B30",
                    symbol="x",
                    line=dict(width=1.2),
                ),
                hovertemplate=(
                    f"{poi.poi_name}"
                    "<extra></extra>"
                ),
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.update_xaxes(
            range=[
                display_bounds[0],
                display_bounds[2],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

        figure.update_yaxes(
            range=[
                display_bounds[1],
                display_bounds[3],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

    # --------------------------------------------------------
    # ROW 2. SPATIAL COMPLETENESS AS GREEN HEATMAP
    # --------------------------------------------------------

    figure.add_trace(
        go.Heatmap(
            x=four_day["date_start"],
            y=["Spatial completeness"],
            z=[
                four_day[
                    "spatial_coverage_pct"
                ].to_numpy()
            ],
            coloraxis="coloraxis2",
            zsmooth=False,
            hoverongaps=False,
            xgap=0,
            ygap=0,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            name="Spatial completeness",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # --------------------------------------------------------
    # ROW 3. DAILY AND FOUR-DAY RECOVERY
    # --------------------------------------------------------

    figure.add_trace(
        go.Scatter(
            x=daily["date_start"],
            y=daily["recovery_pct"],
            mode="lines+markers",
            name="Daily RQ NTL",
            connectgaps=False,
            line=dict(
                color="#63BE7B",
                width=1.0,
            ),
            marker=dict(
                size=2.7,
            ),
            opacity=0.55,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Daily: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    error_plus = (
        four_day["recovery_high_pct"]
        - four_day["recovery_pct"]
    ).clip(lower=0)

    error_minus = (
        four_day["recovery_pct"]
        - four_day["recovery_low_pct"]
    ).clip(lower=0)

    figure.add_trace(
        go.Scatter(
            x=four_day["date_start"],
            y=four_day["recovery_pct"],
            mode="lines+markers",
            name="RQ NTL (4D) ± baseline IQR",
            connectgaps=False,
            line=dict(
                color="#008F3D",
                width=2.7,
                shape="hv",
            ),
            marker=dict(
                size=5,
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color="#006C2E",
                thickness=1.1,
                width=2,
            ),
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------

    figure.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.2,
            dash="dot",
        ),
        row=3,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=2,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # FORCE IDENTICAL SC AND TIME-SERIES X AXES
    # --------------------------------------------------------

    time_start = min(
        daily["date_start"].min(),
        four_day["date_start"].min(),
    )

    time_end = max(
        daily["date_start"].max(),
        four_day["date_start"].max(),
    )

    time_range = [
        time_start,
        time_end,
    ]

    # Row 1 creates len(stage_names) x-axes.
    # The SC panel is therefore the next axis.
    sc_xaxis_reference = (
        f"x{len(stage_names) + 1}"
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        matches=sc_xaxis_reference,
        title_text="Date",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    figure.update_yaxes(
        title_text="SC",
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_yaxes(
        title_text="Recovery<br>(% baseline)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1600,
        height=900,

        title=dict(
            text=(
                f"{poi.poi_name} | {poi.rationale}"
                "<br>"
                f"<sup>{POI_KERNEL_SIZE}×"
                f"{POI_KERNEL_SIZE} VIIRS support; "
                f"{SETTLEMENT_MASK}; "
                f"SC ≥ "
                f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
                "</sup>"
            ),
            x=0.5,
            xanchor="center",
        ),

        coloraxis=dict(
            colorscale=[
                [0.00, "#8E1B1B"],
                [0.35, "#E67E22"],
                [0.70, "#F4D03F"],
                [100 / 140, "#F5F5F5"],
                [0.86, "#00F7FF"],
                [1.00, "#4800FF"],
            ],
            cmin=0,
            cmax=140,
            colorbar=dict(
                title=(
                    "Recovery<br>"
                    "(% baseline)"
                ),
                tickvals=[
                    0,
                    50,
                    80,
                    100,
                    120,
                    140,
                ],
                x=1.005,
                y=0.79,
                len=0.40,
                thickness=16,
            ),
        ),

        coloraxis2=dict(
            colorscale=SC_GREEN_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title="SC (%)",
                tickvals=[
                    0,
                    100,
                ],
                ticktext=[
                    "0",
                    "100",
                ],
                x=1.005,
                y=0.48,
                len=0.14,
                thickness=16,
            ),
        ),

        legend=dict(
            orientation="h",
            y=0.3,
            x=0.5,
        ),

        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),

        margin=dict(
            l=90,
            r=125,
            t=125,
            b=65,
        ),

        hovermode="x unified",
    )

    poi_story_figures[poi.poi_name] = figure
    figure.show()

## 4. Municipality profiles: every location, no selector

Each municipality is shown as a paired panel: spatial completeness above, four-day recovery below. Error bars retain the baseline-sensitivity information without using a shaded envelope. Missing segments remain disconnected.

In [ ]:
# ============================================================
# FIGURE 4. STATIC MUNICIPALITY PROFILE GRID
# 4 columns × 3 municipality rows
# Each municipality contains a short SC strip above its profile
# ============================================================

municipal_units = sorted(
    municipal_four_day["unit_name"].unique()
)

profile_columns = 4
profile_groups = int(
    np.ceil(
        len(municipal_units)
        / profile_columns
    )
)

# Two technical subplot rows per municipality row:
# one narrow SC strip and one recovery panel.
profile_rows = profile_groups * 2

SC_GREEN_COLORSCALE = [
    [0.00, "rgba(255,255,255,0.00)"],
    [0.20, "rgba(220,242,215,0.25)"],
    [0.40, "rgba(166,219,160,0.45)"],
    [0.60, "rgba(90,174,110,0.65)"],
    [0.80, "rgba(25,130,70,0.82)"],
    [1.00, "rgba(0,88,43,0.98)"],
]

# Use one common date range so every SC strip aligns with its
# corresponding recovery time series and all municipalities
# remain directly comparable.
profile_time_range = [
    municipal_four_day["date_start"].min(),
    municipal_four_day["date_start"].max(),
]


def subplot_axis_reference(
    axis_prefix,
    row,
    column,
    number_of_columns,
):
    axis_number = (
        (row - 1) * number_of_columns
        + column
    )

    return (
        axis_prefix
        if axis_number == 1
        else f"{axis_prefix}{axis_number}"
    )


fig_municipal_grid = make_subplots(
    rows=profile_rows,
    cols=profile_columns,
    shared_xaxes=False,
    vertical_spacing=0.025,
    horizontal_spacing=0.045,

    # The SC rows are deliberately short because SC is encoded
    # by colour intensity rather than bar height.
    row_heights=[
        0.10,
        0.90,
    ]
    * profile_groups,
)

for unit_index, unit_name in enumerate(
    municipal_units
):

    group_index = (
        unit_index
        // profile_columns
    )

    column = (
        unit_index
        % profile_columns
        + 1
    )

    coverage_row = (
        group_index * 2
        + 1
    )

    recovery_row = (
        coverage_row
        + 1
    )

    profile = (
        municipal_four_day.loc[
            municipal_four_day[
                "unit_name"
            ]
            == unit_name
        ]
        .sort_values("date_start")
        .copy()
    )

    coverage_xaxis_reference = (
        subplot_axis_reference(
            "x",
            coverage_row,
            column,
            profile_columns,
        )
    )

    recovery_xaxis_reference = (
        subplot_axis_reference(
            "x",
            recovery_row,
            column,
            profile_columns,
        )
    )

    recovery_yaxis_reference = (
        subplot_axis_reference(
            "y",
            recovery_row,
            column,
            profile_columns,
        )
    )

    # --------------------------------------------------------
    # SPATIAL-COMPLETENESS COLOUR STRIP
    # --------------------------------------------------------

    fig_municipal_grid.add_trace(
        go.Heatmap(
            x=profile["date_start"],
            y=["SC"],
            z=[
                profile[
                    "spatial_coverage_pct"
                ].to_numpy()
            ],
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            xgap=0,
            ygap=0,
            hovertemplate=(
                f"<b>{unit_name}</b>"
                "<br>%{x|%d %b %Y}"
                "<br>SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            name="Spatial completeness",
            showlegend=False,
        ),
        row=coverage_row,
        col=column,
    )

    # --------------------------------------------------------
    # FOUR-DAY RECOVERY PROFILE
    # --------------------------------------------------------

    error_plus = (
        profile["recovery_high_pct"]
        - profile["recovery_pct"]
    ).clip(lower=0)

    error_minus = (
        profile["recovery_pct"]
        - profile["recovery_low_pct"]
    ).clip(lower=0)

    fig_municipal_grid.add_trace(
        go.Scatter(
            x=profile["date_start"],
            y=profile["recovery_pct"],
            mode="lines+markers",
            connectgaps=False,
            line=dict(
                color="#008F3D",
                width=2.6,
                shape="hv",
            ),
            marker=dict(
                size=4.5,
                color="#008F3D",
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color="#006C2E",
                thickness=1.1,
                width=2,
            ),
            name="RQ NTL (4D) ± baseline IQR",
            showlegend=(
                unit_index == 0
            ),
            hovertemplate=(
                f"<b>{unit_name}</b>"
                "<br>%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=recovery_row,
        col=column,
    )

    # --------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------

    fig_municipal_grid.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.2,
            dash="dot",
        ),
        row=recovery_row,
        col=column,
    )

    fig_municipal_grid.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=1.8,
            dash="dash",
        ),
        row=coverage_row,
        col=column,
    )

    fig_municipal_grid.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=1.8,
            dash="dash",
        ),
        row=recovery_row,
        col=column,
    )

    # --------------------------------------------------------
    # MUNICIPALITY LABEL
    # --------------------------------------------------------

    fig_municipal_grid.add_annotation(
        x=0.015,
        y=0.97,
        xref=(
            f"{recovery_xaxis_reference} "
            "domain"
        ),
        yref=(
            f"{recovery_yaxis_reference} "
            "domain"
        ),
        text=f"<b>{unit_name}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.82)",
        borderpad=3,
        font=dict(
            family="Arial",
            size=18,
            color="#243B5A",
        ),
    )

    # --------------------------------------------------------
    # IDENTICAL SC AND RECOVERY X AXES
    # --------------------------------------------------------

    fig_municipal_grid.update_xaxes(
        range=profile_time_range,
        autorange=False,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=coverage_row,
        col=column,
    )

    fig_municipal_grid.update_xaxes(
        range=profile_time_range,
        autorange=False,
        matches=coverage_xaxis_reference,
        showticklabels=(
            group_index
            == profile_groups - 1
        ),
        title_text=(
            "Date"
            if group_index
            == profile_groups - 1
            else None
        ),
        title_font=dict(
            size=18,
        ),
        tickfont=dict(
            size=14,
        ),
        tickformat="%b<br>%Y",
        nticks=5,
        showgrid=True,
        gridcolor="#E8EDF3",
        gridwidth=1,
        zeroline=False,
        row=recovery_row,
        col=column,
    )

    # --------------------------------------------------------
    # Y AXES
    # --------------------------------------------------------

    fig_municipal_grid.update_yaxes(
        title_text=(
            "SC"
            if column == 1
            else None
        ),
        title_font=dict(
            size=17,
        ),
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=coverage_row,
        col=column,
    )

    fig_municipal_grid.update_yaxes(
        rangemode="tozero",
        showticklabels=(
            column == 1
        ),
        title_text=(
            "Recovery (%)"
            if column == 1
            else None
        ),
        title_font=dict(
            size=17,
        ),
        tickfont=dict(
            size=14,
        ),
        showgrid=True,
        gridcolor="#E8EDF3",
        gridwidth=1,
        zeroline=False,
        row=recovery_row,
        col=column,
    )


# ============================================================
# HIDE UNUSED PANELS IF THE NUMBER OF MUNICIPALITIES IS < 12
# ============================================================

total_profile_slots = (
    profile_columns
    * profile_groups
)

for empty_index in range(
    len(municipal_units),
    total_profile_slots,
):

    empty_group = (
        empty_index
        // profile_columns
    )

    empty_column = (
        empty_index
        % profile_columns
        + 1
    )

    empty_coverage_row = (
        empty_group * 2
        + 1
    )

    empty_recovery_row = (
        empty_coverage_row
        + 1
    )

    fig_municipal_grid.update_xaxes(
        visible=False,
        row=empty_coverage_row,
        col=empty_column,
    )

    fig_municipal_grid.update_yaxes(
        visible=False,
        row=empty_coverage_row,
        col=empty_column,
    )

    fig_municipal_grid.update_xaxes(
        visible=False,
        row=empty_recovery_row,
        col=empty_column,
    )

    fig_municipal_grid.update_yaxes(
        visible=False,
        row=empty_recovery_row,
        col=empty_column,
    )


# ============================================================
# LAYOUT
# ============================================================

fig_municipal_grid.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    # 16:9 presentation format.
    width=1600,
    height=900,
    coloraxis=dict(
        colorscale=SC_GREEN_COLORSCALE,
        cmin=0,
        cmax=100,
        colorbar=dict(
            title=dict(
                text="SC (%)",
                font=dict(
                    size=17,
                ),
            ),
            tickvals=[
                0,
                100,
            ],
            ticktext=[
                "0",
                "100",
            ],
            tickfont=dict(
                size=14,
            ),
            x=1.008,
            y=1,
            len=0.10,
            thickness=20,
            outlinewidth=1,
        ),
    ),

    legend=dict(
        orientation="h",
        x=0.8,
        y=0.32,
        font=dict(
            size=17,
        ),
    ),

    font=dict(
        family="Arial",
        size=15,
        color="#243B5A",
    ),

    margin=dict(
        l=105,
        r=105,
        t=110,
        b=75,
    ),

    hovermode="x unified",
)

fig_municipal_grid.show()

## 5. One centralized profile for all municipalities

The completeness heatmap shows when each municipality was directly observed. The lower panel overlays all municipal recovery curves, the cross-location median with inter-location IQR error bars, the regional RQ bridge, and NGCP demand. This is a comparison of trajectories, not evidence that every LGU followed the sub-grid load.

In [ ]:
# ============================================================
# FIGURE 5. CENTRALIZED MULTI-LOCATION PROFILE
# ============================================================

SC_GREEN_COLORSCALE = [
    [0.00, "rgba(255,255,255,0.00)"],
    [0.20, "rgba(220,242,215,0.25)"],
    [0.40, "rgba(166,219,160,0.45)"],
    [0.60, "rgba(90,174,110,0.65)"],
    [0.80, "rgba(25,130,70,0.82)"],
    [1.00, "rgba(0,88,43,0.98)"],
]

coverage_matrix = (
    municipal_four_day
    .pivot(
        index="unit_name",
        columns="date_start",
        values="spatial_coverage_pct",
    )
    .reindex(municipal_units)
)

recovery_matrix = (
    municipal_four_day
    .pivot(
        index="unit_name",
        columns="date_start",
        values="recovery_pct",
    )
    .reindex(municipal_units)
)

central_dates = recovery_matrix.columns

central_median = recovery_matrix.median(
    axis=0,
    skipna=True,
)

central_q25 = recovery_matrix.quantile(
    0.25,
    axis=0,
)

central_q75 = recovery_matrix.quantile(
    0.75,
    axis=0,
)

fig_central_profile = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[
        0.38,
        0.62,
    ],
    subplot_titles=(
        "Direct-observation spatial completeness",
        "Municipality recovery profiles and regional bridge",
    ),
)

# ------------------------------------------------------------
# ROW 1. GREEN SPATIAL-COMPLETENESS HEATMAP
# ------------------------------------------------------------

fig_central_profile.add_trace(
    go.Heatmap(
        x=coverage_matrix.columns,
        y=coverage_matrix.index,
        z=coverage_matrix.values,
        colorscale=SC_GREEN_COLORSCALE,
        zmin=0,
        zmax=100,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            title=dict(
                text="SC (%)",
                font=dict(
                    size=19,
                ),
            ),
            tickvals=[
                0,
                SPATIAL_COMPLETENESS_PCT,
                50,
                100,
            ],
            ticktext=[
                "0",
                f"{SPATIAL_COMPLETENESS_PCT:.0f}",
                "50",
                "100",
            ],
            tickfont=dict(
                size=16,
            ),
            y=0.81,
            len=0.33,
            thickness=19,
            outlinewidth=1,
        ),
        hovertemplate=(
            "<b>%{y}</b>"
            "<br>%{x|%d %b %Y}"
            "<br>SC: %{z:.1f}%"
            "<extra></extra>"
        ),
        name="Spatial completeness",
    ),
    row=1,
    col=1,
)

# ------------------------------------------------------------
# ROW 2. INDIVIDUAL MUNICIPALITY PROFILES
# ------------------------------------------------------------

palette = (
    px.colors.qualitative.Safe
    + px.colors.qualitative.Set2
)

for unit_index, unit_name in enumerate(
    municipal_units
):

    profile = (
        municipal_four_day.loc[
            municipal_four_day[
                "unit_name"
            ]
            == unit_name
        ]
        .sort_values("date_start")
        .copy()
    )

    fig_central_profile.add_trace(
        go.Scatter(
            x=profile["date_start"],
            y=profile["recovery_pct"],
            mode="lines",
            name=unit_name,
            connectgaps=False,
            line=dict(
                color=palette[
                    unit_index
                    % len(palette)
                ],
                width=1.6,
            ),
            opacity=0.72,
            hovertemplate=(
                f"<b>{unit_name}</b>"
                "<br>%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# ------------------------------------------------------------
# MUNICIPALITY MEDIAN WITH INTER-LOCATION IQR
# ------------------------------------------------------------

fig_central_profile.add_trace(
    go.Scatter(
        x=central_dates,
        y=central_median,
        mode="lines+markers",
        name=(
            "Median across municipalities "
            "± IQR"
        ),
        connectgaps=False,
        line=dict(
            color="#008F3D",
            width=4,
        ),
        marker=dict(
            size=7,
            color="#008F3D",
        ),
        error_y=dict(
            type="data",
            symmetric=False,
            array=(
                central_q75
                - central_median
            ).clip(lower=0),
            arrayminus=(
                central_median
                - central_q25
            ).clip(lower=0),
            color="#006C2E",
            thickness=1.5,
            width=3,
        ),
        hovertemplate=(
            "<b>Municipality median</b>"
            "<br>%{x|%d %b %Y}"
            "<br>Recovery: %{y:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

# ------------------------------------------------------------
# BOLDER REGIONAL RQ NTL
# ------------------------------------------------------------

fig_central_profile.add_trace(
    go.Scatter(
        x=regional_four_day["date_start"],
        y=regional_four_day["recovery_pct"],
        mode="lines",
        name="Regional RQ NTL",
        connectgaps=False,
        line=dict(
            color="#002FFF",
            width=5.5,
        ),
        hovertemplate=(
            "<b>Regional RQ NTL</b>"
            "<br>%{x|%d %b %Y}"
            "<br>Recovery: %{y:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

# ------------------------------------------------------------
# BOLDER NGCP PROFILE
# ------------------------------------------------------------

fig_central_profile.add_trace(
    go.Scatter(
        x=ngcp_four_day["date_start"],
        y=ngcp_four_day["recovery_pct"],
        mode="lines",
        name="NGCP 01:00 load",
        connectgaps=False,
        line=dict(
            color="#111111",
            width=5,
            dash="dash",
        ),
        hovertemplate=(
            "<b>NGCP 01:00 load</b>"
            "<br>%{x|%d %b %Y}"
            "<br>Recovery: %{y:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

# ------------------------------------------------------------
# REFERENCE LINES
# ------------------------------------------------------------

fig_central_profile.add_hline(
    y=100,
    line=dict(
        color="#7F8C8D",
        width=1.7,
        dash="dot",
    ),
    row=2,
    col=1,
)

fig_central_profile.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color="#0057FF",
        width=2.8,
        dash="dash",
    ),
    row=1,
    col=1,
)

fig_central_profile.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color="#0057FF",
        width=2.8,
        dash="dash",
    ),
    row=2,
    col=1,
)

# ------------------------------------------------------------
# AXES
# ------------------------------------------------------------

fig_central_profile.update_yaxes(
    title_text="Location",
    title_font=dict(size=21),
    tickmode="array",
    tickvals=coverage_matrix.index.tolist(),
    ticktext=coverage_matrix.index.tolist(),
    tickfont=dict(size=14),
    showticklabels=True,
    automargin=True,
    showgrid=False,
    zeroline=False,
    row=1,
    col=1,
)

fig_central_profile.update_xaxes(
    title_text="Date",
    title_font=dict(
        size=22,
    ),
    tickfont=dict(
        size=17,
    ),
    tickformat="%b<br>%Y",
    nticks=10,
    showgrid=True,
    gridcolor="#E8EDF3",
    gridwidth=1,
    zeroline=False,
    row=2,
    col=1,
)

fig_central_profile.update_yaxes(
    title_text="Location",
    title_font=dict(
        size=21,
    ),
    tickfont=dict(
        size=16,
    ),
    showgrid=False,
    zeroline=False,
    row=1,
    col=1,
)

fig_central_profile.update_yaxes(
    title_text=(
        "Recovery relative to "
        "baseline (%)"
    ),
    title_font=dict(
        size=21,
    ),
    tickfont=dict(
        size=17,
    ),
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E8EDF3",
    gridwidth=1,
    zeroline=False,
    row=2,
    col=1,
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig_central_profile.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    # 16:9 presentation format.
    width=1600,
    height=900,
    legend=dict(
        orientation="h",
        y=-0.18,
        x=0,
        font=dict(
            size=15,
        ),
        bgcolor="rgba(255,255,255,0.80)",
    ),

    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),

    margin=dict(
        l=155,
        r=115,
        t=115,
        b=180,
    ),

    hovermode="x unified",
)

fig_central_profile.update_annotations(
    font=dict(
        family="Arial",
        size=21,
        color="#243B5A",
    )
)

fig_central_profile.show()

display(
    Markdown(
        f"**Regional bridge decision:** "
        f"{bridge_decision}"
    )
)

## 6. Recovery milestones instead of result tables

Impact-drop error bars reflect baseline sensitivity. T50, T80, and T90 points show the first persistent supporting composite; horizontal bars extend back to the last admissible below-threshold observation. Missing milestones are not plotted and must be read with the quality flag in the compact summary.

In [ ]:
# ============================================================
# FIGURE 6A. OBSERVED IMPACT DROP
# ============================================================


def parse_numeric_range(value):
    if value is None or pd.isna(value):
        return (np.nan, np.nan)
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", str(value))
    if len(numbers) < 2:
        return (np.nan, np.nan)
    return float(numbers[0]), float(numbers[1])


metric_frames = [municipal_metrics.assign(support_type="Municipality")]
if not poi_metrics.empty:
    metric_frames.append(poi_metrics.assign(support_type="POI kernel"))

all_location_metrics = pd.concat(metric_frames, ignore_index=True)
all_location_metrics["display_name"] = (
    all_location_metrics["unit_name"]
    + np.where(all_location_metrics["support_type"].eq("POI kernel"), " · POI", "")
)

impact_plot = all_location_metrics.sort_values("impact_drop_pct", ascending=True).copy()
impact_ranges = impact_plot["impact_drop_range_pct"].map(parse_numeric_range)
impact_low = np.array([value[0] for value in impact_ranges], dtype=float)
impact_high = np.array([value[1] for value in impact_ranges], dtype=float)

quality_colors = {
    "OBS_OK": "#168A43",
    "OBS_LIMITED": "#E69F00",
    "NOT_OBSERVABLE": "#9E9E9E",
}

fig_impact = go.Figure()
fig_impact.add_trace(
    go.Bar(
        x=impact_plot["impact_drop_pct"],
        y=impact_plot["display_name"],
        orientation="h",
        marker_color=impact_plot["quality_flag"].map(quality_colors),
        error_x=dict(
            type="data",
            symmetric=False,
            array=np.maximum(0, impact_high - impact_plot["impact_drop_pct"].to_numpy()),
            arrayminus=np.maximum(0, impact_plot["impact_drop_pct"].to_numpy() - impact_low),
            color="#3B3B3B",
            thickness=1.1,
            width=3,
        ),
        text=impact_plot["quality_flag"] + " · " + impact_plot["support_type"],
        hovertemplate=(
            "%{y}<br>Observed impact drop: %{x:.1f}%<br>"
            "%{text}<extra></extra>"
        ),
    )
)
fig_impact.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    width=1150,
    height=max(520, 42 * len(impact_plot)),
    title="Deepest observed four-day NTL drop within 60 days of Haiyan",
    xaxis_title="Drop from baseline (%)",
    yaxis_title=None,
    font=dict(family="Arial", size=12, color="#243B5A"),
    margin=dict(l=210, r=45, t=85, b=65),
)
fig_impact.show()

In [ ]:
# ============================================================
# FIGURE 6B. T50, T80, AND T90
# SEPARATE MUNICIPALITY AND POI PANELS
# ============================================================


def interval_lower_day(value):
    if value is None or pd.isna(value):
        return np.nan

    numbers = re.findall(
        r"\d+",
        str(value),
    )

    return (
        float(numbers[0])
        if numbers
        else np.nan
    )


milestone_colors = {
    50: "#56B4E9",
    80: "#009E73",
    90: "#D55E00",
}

milestone_symbols = {
    50: "circle",
    80: "square",
    90: "diamond",
}

municipality_milestones = (
    all_location_metrics.loc[
        all_location_metrics[
            "support_type"
        ]
        == "Municipality"
    ]
    .sort_values("unit_name")
    .copy()
)

poi_milestones = (
    all_location_metrics.loc[
        all_location_metrics[
            "support_type"
        ]
        == "POI kernel"
    ]
    .sort_values("unit_name")
    .copy()
)

panel_data = {
    1: {
        "title": "Cities and municipalities",
        "data": municipality_milestones,
    },
    2: {
        "title": "Characteristic POI kernels",
        "data": poi_milestones,
    },
}

analysis_end_day = int(
    (
        PROFILE_END
        - EVENT_DATE
    ).days
)

fig_milestones = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.2,
    subplot_titles=(
        "Cities and municipalities",
        "Characteristic POI kernels",
    ),
)

for panel_column, panel in panel_data.items():

    panel_metrics = panel["data"].copy()

    location_order = (
        panel_metrics["unit_name"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    # Plotly categorical axes run from bottom to top.
    # Reverse the array so alphabetical order reads top to bottom.
    category_order = location_order[::-1]

    # Invisible points ensure locations without an identifiable
    # milestone remain present on the y-axis.
    fig_milestones.add_trace(
        go.Scatter(
            x=np.zeros(
                len(location_order)
            ),
            y=location_order,
            mode="markers",
            marker=dict(
                size=1,
                color="rgba(0,0,0,0)",
            ),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=panel_column,
    )

    for threshold in (
        50,
        80,
        90,
    ):

        day_column = (
            f"T{threshold}_day"
        )

        interval_column = (
            f"T{threshold}_"
            "observation_interval"
        )

        status_column = (
            f"T{threshold}_status"
        )

        available = panel_metrics.loc[
            panel_metrics[
                day_column
            ].notna()
        ].copy()

        if available.empty:
            continue

        point_days = (
            available[day_column]
            .to_numpy(dtype=float)
        )

        lower_days = (
            available[interval_column]
            .map(interval_lower_day)
            .to_numpy(dtype=float)
        )

        # Fall back to zero-width intervals when the lower
        # observation bound is unavailable.
        lower_days = np.where(
            np.isfinite(lower_days),
            lower_days,
            point_days,
        )

        interval_width = np.maximum(
            0,
            point_days
            - lower_days,
        )

        customdata = np.column_stack(
            [
                available[
                    interval_column
                ]
                .fillna("—")
                .astype(str),
                available[
                    status_column
                ]
                .fillna("—")
                .astype(str),
            ]
        )

        fig_milestones.add_trace(
            go.Scatter(
                x=point_days,
                y=available["unit_name"],
                mode="markers",
                name=f"T{threshold}",
                legendgroup=f"T{threshold}",
                showlegend=(
                    panel_column == 1
                ),
                marker=dict(
                    size=13,
                    color=milestone_colors[
                        threshold
                    ],
                    symbol=milestone_symbols[
                        threshold
                    ],
                    line=dict(
                        color="white",
                        width=1.3,
                    ),
                ),
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=np.zeros(
                        len(available)
                    ),
                    arrayminus=interval_width,
                    color=milestone_colors[
                        threshold
                    ],
                    thickness=2,
                    width=4,
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{y}</b>"
                    f"<br>T{threshold}: "
                    "%{x:.0f} days"
                    "<br>Observation interval: "
                    "%{customdata[0]}"
                    "<br>Status: "
                    "%{customdata[1]}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=panel_column,
        )

    # --------------------------------------------------------
    # RECOVERY-STAGE BOUNDARIES
    # --------------------------------------------------------

    for boundary_day in (
        60,
        120,
        180,
    ):

        fig_milestones.add_vline(
            x=boundary_day,
            line=dict(
                color="#AAB6C2",
                width=1.3,
                dash="dot",
            ),
            row=1,
            col=panel_column,
        )

    # --------------------------------------------------------
    # PANEL AXES
    # --------------------------------------------------------

    fig_milestones.update_xaxes(
        range=[
            0,
            analysis_end_day,
        ],
        title_text=(
            "Days after Haiyan"
        ),
        title_font=dict(
            size=20,
        ),
        tickfont=dict(
            size=16,
        ),
        tickvals=[
            0,
            60,
            120,
            180,
            240,
            300,
            360,
        ],
        showgrid=True,
        gridcolor="#E8EDF3",
        gridwidth=1,
        zeroline=False,
        row=1,
        col=panel_column,
    )

    fig_milestones.update_yaxes(
        categoryorder="array",
        categoryarray=category_order,
        tickmode="array",
        tickvals=location_order,
        ticktext=location_order,
        tickfont=dict(
            size=16,
        ),
        automargin=True,
        showgrid=True,
        gridcolor="#EEF2F6",
        gridwidth=1,
        zeroline=False,
        row=1,
        col=panel_column,
    )


# ============================================================
# STAGE LABELS
# ============================================================

stage_labels = [
    {
        "day": 30,
        "label": "Stage 1",
    },
    {
        "day": 90,
        "label": "Stage 2",
    },
    {
        "day": 150,
        "label": "Stage 3",
    },
    {
        "day": 270,
        "label": "Extended",
    },
]

for panel_column in (
    1,
    2,
):

    axis_reference = (
        "x"
        if panel_column == 1
        else "x2"
    )

    y_reference = (
        "y domain"
        if panel_column == 1
        else "y2 domain"
    )

    for stage in stage_labels:

        fig_milestones.add_annotation(
            x=stage["day"],
            y=1,
            xref=axis_reference,
            yref=y_reference,
            text=stage["label"],
            showarrow=False,
            font=dict(
                family="Arial",
                size=14,
                color="#657789",
            ),
        )


# ============================================================
# LAYOUT
# ============================================================

fig_milestones.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    # Presentation-ready 16:9 landscape.
    width=1600,
    height=800,

    legend=dict(
        orientation="h",
        x=0.5,
        y=1.075,
        xanchor="center",
        yanchor="bottom",
        font=dict(
            size=17,
        ),
        bgcolor="rgba(255,255,255,0.85)",
        itemsizing="constant",
    ),

    font=dict(
        family="Arial",
        size=16,
        color="#243B5A",
    ),

    margin=dict(
        l=210,
        r=80,
        t=145,
        b=85,
    ),

    hovermode="closest",
)

fig_milestones.update_annotations(
    font=dict(
        family="Arial",
        size=20,
        color="#243B5A",
    )
)

fig_milestones.show()

In [ ]:
# ============================================================
# CLEAN FOUR-DAY RECOVERY PROFILES
# 1. Municipalities: 4 columns × 3 rows
# 2. POI kernels: 3 columns × 2 rows
# ============================================================

MILESTONE_COLORS = {
    50: "#56B4E9",
    80: "#009E73",
    90: "#D55E00",
}

MILESTONE_SYMBOLS = {
    50: "circle",
    80: "square",
    90: "diamond",
}

MILESTONE_TEXT_POSITIONS = {
    50: "bottom center",
    80: "top left",
    90: "top right",
}


def metric_day_label(value):
    if value is None or pd.isna(value):
        return "—"

    return f"{int(round(float(value)))} d"


def build_clean_recovery_grid(
    profiles,
    metrics,
    location_order,
    columns,
    rows,
    figure_title,
    line_color="#008F3D",
):

    location_order = [
        location
        for location in location_order
        if location
        in set(
            profiles["unit_name"].unique()
        )
    ]

    maximum_slots = columns * rows

    if len(location_order) > maximum_slots:
        raise ValueError(
            f"{len(location_order)} locations cannot fit "
            f"inside a {columns} × {rows} grid."
        )

    subplot_titles = (
        location_order
        + [""] * (
            maximum_slots
            - len(location_order)
        )
    )

    figure = make_subplots(
        rows=rows,
        cols=columns,
        horizontal_spacing=0.055,
        vertical_spacing=0.095,
        subplot_titles=subplot_titles,
    )

    metric_lookup = (
        metrics
        .drop_duplicates("unit_name")
        .set_index("unit_name")
        if not metrics.empty
        else pd.DataFrame()
    )

    # Use identical temporal and radiance ranges in every panel.
    profile_start = profiles[
        "date_start"
    ].min()

    profile_end = profiles[
        "date_start"
    ].max()

    finite_recovery = pd.to_numeric(
        profiles["recovery_pct"],
        errors="coerce",
    ).dropna()

    if finite_recovery.empty:
        common_y_min = 0
        common_y_max = 140
    else:
        common_y_min = min(
            0,
            float(
                finite_recovery.min()
            )
            * 1.05,
        )

        common_y_max = max(
            140,
            float(
                finite_recovery.max()
            )
            * 1.08,
        )

    for location_index, location_name in enumerate(
        location_order
    ):

        row = (
            location_index
            // columns
            + 1
        )

        column = (
            location_index
            % columns
            + 1
        )

        profile = (
            profiles.loc[
                profiles["unit_name"]
                == location_name
            ]
            .sort_values("date_start")
            .copy()
        )

        # ----------------------------------------------------
        # FOUR-DAY RELIABILITY-QUALIFIED NTL TRAJECTORY
        # ----------------------------------------------------

        figure.add_trace(
            go.Scatter(
                x=profile["date_start"],
                y=profile["recovery_pct"],
                mode="lines+markers",
                name="RQ NTL (4D)",
                legendgroup="rq_ntl",
                showlegend=(
                    location_index == 0
                ),
                connectgaps=False,
                line=dict(
                    color=line_color,
                    width=2.8,
                    shape="hv",
                ),
                marker=dict(
                    size=4.5,
                    color=line_color,
                ),
                hovertemplate=(
                    f"<b>{location_name}</b>"
                    "<br>%{x|%d %b %Y}"
                    "<br>Recovery: %{y:.1f}%"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=column,
        )

        # ----------------------------------------------------
        # BASELINE AND HAIYAN REFERENCES
        # ----------------------------------------------------

        figure.add_hline(
            y=100,
            line=dict(
                color="#7F8C8D",
                width=1.2,
                dash="dot",
            ),
            row=row,
            col=column,
        )

        figure.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line=dict(
                color="#0057FF",
                width=1.8,
                dash="dash",
            ),
            row=row,
            col=column,
        )

        # ----------------------------------------------------
        # T50, T80, AND T90
        # ----------------------------------------------------

        milestone_summary = []

        if (
            not metric_lookup.empty
            and location_name
            in metric_lookup.index
        ):

            location_metrics = (
                metric_lookup.loc[
                    location_name
                ]
            )

            for threshold in (
                50,
                80,
                90,
            ):

                day_column = (
                    f"T{threshold}_day"
                )

                status_column = (
                    f"T{threshold}_status"
                )

                milestone_day = (
                    location_metrics.get(
                        day_column,
                        np.nan,
                    )
                )

                milestone_status = (
                    location_metrics.get(
                        status_column,
                        "—",
                    )
                )

                milestone_summary.append(
                    f"T{threshold}: "
                    f"{metric_day_label(milestone_day)}"
                )

                if pd.isna(milestone_day):
                    continue

                milestone_date = (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=float(
                            milestone_day
                        )
                    )
                )

                observed_profile = (
                    profile.dropna(
                        subset=[
                            "recovery_pct"
                        ]
                    )
                )

                if observed_profile.empty:
                    continue

                nearest_index = (
                    (
                        observed_profile[
                            "date_start"
                        ]
                        - milestone_date
                    )
                    .abs()
                    .idxmin()
                )

                milestone_recovery = float(
                    observed_profile.loc[
                        nearest_index,
                        "recovery_pct",
                    ]
                )

                figure.add_trace(
                    go.Scatter(
                        x=[
                            milestone_date
                        ],
                        y=[
                            milestone_recovery
                        ],
                        mode="markers+text",
                        text=[
                            f"T{threshold}"
                        ],
                        textposition=(
                            MILESTONE_TEXT_POSITIONS[
                                threshold
                            ]
                        ),
                        textfont=dict(
                            size=13,
                            color=(
                                MILESTONE_COLORS[
                                    threshold
                                ]
                            ),
                        ),
                        marker=dict(
                            size=11,
                            color=(
                                MILESTONE_COLORS[
                                    threshold
                                ]
                            ),
                            symbol=(
                                MILESTONE_SYMBOLS[
                                    threshold
                                ]
                            ),
                            line=dict(
                                color="white",
                                width=1.2,
                            ),
                        ),
                        name=f"T{threshold}",
                        legendgroup=(
                            f"T{threshold}"
                        ),
                        showlegend=(
                            location_index == 0
                        ),
                        customdata=[
                            [
                                milestone_day,
                                milestone_status,
                            ]
                        ],
                        hovertemplate=(
                            f"<b>{location_name}</b>"
                            f"<br>T{threshold}: "
                            "%{customdata[0]:.0f} days"
                            "<br>Observed recovery: "
                            "%{y:.1f}%"
                            "<br>Status: "
                            "%{customdata[1]}"
                            "<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=column,
                )

        else:
            milestone_summary = [
                "T50: —",
                "T80: —",
                "T90: —",
            ]

        # ----------------------------------------------------
        # COMPACT MILESTONE SUMMARY
        # ----------------------------------------------------

        xaxis_number = (
            (row - 1)
            * columns
            + column
        )

        xaxis_reference = (
            "x"
            if xaxis_number == 1
            else f"x{xaxis_number}"
        )

        yaxis_reference = (
            "y"
            if xaxis_number == 1
            else f"y{xaxis_number}"
        )

        figure.add_annotation(
            x=0.015,
            y=0.975,
            xref=(
                f"{xaxis_reference} "
                "domain"
            ),
            yref=(
                f"{yaxis_reference} "
                "domain"
            ),
            text="  ·  ".join(
                milestone_summary
            ),
            showarrow=False,
            xanchor="left",
            yanchor="top",
            bgcolor=(
                "rgba(255,255,255,0.82)"
            ),
            borderpad=3,
            font=dict(
                family="Arial",
                size=13,
                color="#243B5A",
            ),
        )

        # ----------------------------------------------------
        # IDENTICAL AXES ACROSS ALL PANELS
        # ----------------------------------------------------

        figure.update_xaxes(
            range=[
                profile_start,
                profile_end,
            ],
            autorange=False,
            showticklabels=(
                row == rows
            ),
            title_text=(
                "Date"
                if row == rows
                else None
            ),
            title_font=dict(
                size=17,
            ),
            tickfont=dict(
                size=13,
            ),
            tickformat="%b<br>%Y",
            nticks=5,
            showgrid=True,
            gridcolor="#E8EDF3",
            gridwidth=1,
            zeroline=False,
            row=row,
            col=column,
        )

        figure.update_yaxes(
            range=[
                common_y_min,
                common_y_max,
            ],
            autorange=False,
            showticklabels=(
                column == 1
            ),
            title_text=(
                "Recovery (%)"
                if column == 1
                else None
            ),
            title_font=dict(
                size=17,
            ),
            tickfont=dict(
                size=13,
            ),
            showgrid=True,
            gridcolor="#E8EDF3",
            gridwidth=1,
            zeroline=False,
            row=row,
            col=column,
        )

    # --------------------------------------------------------
    # HIDE UNUSED SUBPLOTS
    # --------------------------------------------------------

    for empty_index in range(
        len(location_order),
        maximum_slots,
    ):

        empty_row = (
            empty_index
            // columns
            + 1
        )

        empty_column = (
            empty_index
            % columns
            + 1
        )

        figure.update_xaxes(
            visible=False,
            row=empty_row,
            col=empty_column,
        )

        figure.update_yaxes(
            visible=False,
            row=empty_row,
            col=empty_column,
        )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1920,
        height=1080,

        title=dict(
            text=figure_title,
            x=0.5,
            xanchor="center",
            font=dict(
                family="Arial",
                size=28,
                color="#243B5A",
            ),
        ),

        legend=dict(
            orientation="h",
            x=0.5,
            y=1.035,
            xanchor="center",
            yanchor="bottom",
            font=dict(
                size=16,
            ),
            bgcolor=(
                "rgba(255,255,255,0.84)"
            ),
            itemsizing="constant",
        ),

        font=dict(
            family="Arial",
            size=15,
            color="#243B5A",
        ),

        margin=dict(
            l=100,
            r=45,
            t=120,
            b=80,
        ),

        hovermode="closest",
    )

    # Enlarge municipality/POI subplot titles.
    figure.update_annotations(
        font=dict(
            family="Arial",
            size=17,
            color="#243B5A",
        )
    )

    return figure


# ============================================================
# FIGURE 7A. MUNICIPALITY PROFILES — 4 × 3 GRID
# ============================================================

municipality_order = sorted(
    municipal_four_day[
        "unit_name"
    ].unique()
)

fig_municipality_clean_profiles = (
    build_clean_recovery_grid(
        profiles=municipal_four_day,
        metrics=municipal_metrics,
        location_order=municipality_order,
        columns=4,
        rows=3,
        figure_title=(
            "Four-day reliability-qualified NTL recovery: "
            "cities and municipalities"
        ),
        line_color="#008F3D",
    )
)

fig_municipality_clean_profiles.show()


# ============================================================
# FIGURE 7B. POI KERNEL PROFILES — 3 × 2 GRID
# ============================================================

poi_profile_names = set(
    poi_four_day[
        "unit_name"
    ].unique()
)

poi_order = [
    poi_name
    for poi_name
    in poi_table["poi_name"]
    if poi_name
    in poi_profile_names
]

fig_poi_clean_profiles = (
    build_clean_recovery_grid(
        profiles=poi_four_day,
        metrics=poi_metrics,
        location_order=poi_order,
        columns=3,
        rows=2,
        figure_title=(
            f"Four-day reliability-qualified NTL recovery: "
            f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} POI kernels"
        ),
        line_color="#008F3D",
    )
)

fig_poi_clean_profiles.show()

### Compact exact-value summary

The charts remain the primary result. This table is retained only for exact milestone values and the distinction between supported, observation-limited, and not-observable cases.

In [ ]:
# ============================================================
# TABLE 1. COMPACT RECOVERY SUMMARY
# ============================================================

compact_summary = all_location_metrics[
    [
        "support_type",
        "unit_name",
        "quality_flag",
        "retained_pct",
        "max_gap_days",
        "median_event_sc_pct",
        "impact_drop_pct",
        "T50_day",
        "T50_status",
        "T80_day",
        "T80_status",
        "T90_day",
        "T90_status",
    ]
].rename(
    columns={
        "support_type": "Support",
        "unit_name": "Location",
        "quality_flag": "Quality",
        "retained_pct": "Retained (%)",
        "max_gap_days": "Max gap (d)",
        "median_event_sc_pct": "Median SC (%)",
        "impact_drop_pct": "Impact drop (%)",
        "T50_day": "T50 (d)",
        "T50_status": "T50 status",
        "T80_day": "T80 (d)",
        "T80_status": "T80 status",
        "T90_day": "T90 (d)",
        "T90_status": "T90 status",
    }
).sort_values(["Support", "Location"])

display(
    compact_summary.style.format(
        {
            "Retained (%)": "{:.1f}",
            "Median SC (%)": "{:.1f}",
            "Impact drop (%)": "{:.1f}",
            "T50 (d)": "{:.0f}",
            "T80 (d)": "{:.0f}",
            "T90 (d)": "{:.0f}",
        },
        na_rep="—",
    ).hide(axis="index")
)

## 7. Interpretation against documented recovery

- **Tacloban:** compare rapid functional return with severe immediate physical damage, four-year rebuilding, and northward relocation. Functional return and household recovery need not be synchronous ([Sheykhmousa et al., 2019](https://doi.org/10.3390/rs11101174); [Ghaffarian et al., 2021](https://doi.org/10.1016/j.ijdrr.2021.102285)).
- **Palo–Tanauan–Tolosa–Dulag corridor:** strong disruption is consistent with documented storm-surge and distribution-system damage, but smaller illuminated supports make timing more sensitive to observability ([Yi et al., 2015](https://doi.org/10.1016/j.ijdrr.2015.05.007); [JICA, 2015](https://openjicareport.jica.go.jp/pdf/12233953.pdf)).
- **Guiuan and Basey:** weak or absent observed shocks must not be read as limited impact when G2 support or post-landfall observation is inadequate.
- **Ormoc and Baybay:** these are western comparisons, not controls. A smaller or earlier rebound remains compatible with regional grid disruption.

Physical reconstruction, NTL recovery, and household outcomes observe different dimensions. Agreement strengthens interpretation; divergence is a result to explain, not automatically an error.

## 8. Diagnostic comparison: observed versus gap-filled NTL

The focus-location comparison is retained at the end because it diagnoses attenuation and timing changes introduced by gap filling. It is not part of the primary evidence chain.

In [ ]:
# ============================================================
# 16. GAP-FILLED DIAGNOSTIC FOR THE FOCUS LGU
# ============================================================

focus_key = canonical_unit_name(FOCUS_UNIT)
focus_match = selected_raster_crs.loc[selected_raster_crs["unit_key"] == focus_key]

if gap_filled is not None and len(focus_match) == 1:
    focus_row = focus_match.iloc[0]
    focus_support_full = (selected_zone_id == int(focus_row["profile_id"])) & ghsl_mask
    gap_cube_full = gap_filled.where(rq_base_mask)
    focus_gap_cube, focus_gap_support = crop_to_support(gap_cube_full, focus_support_full)

    try:
        (
            focus_gap_profile,
            _,
            _,
            _,
            focus_gap_report,
        ) = build_pixel_matched_profile(
            cube=focus_gap_cube,
            support_mask=focus_gap_support,
            aggregation_days=4,
            unit_name=focus_row["unit_name"],
            unit_type="City/municipality",
            method="Gap-filled diagnostic",
        )

        focus_rq = municipal_four_day.loc[
            municipal_four_day["unit_name"] == focus_row["unit_name"]
        ]

        fig_gap_diagnostic = go.Figure()
        fig_gap_diagnostic.add_trace(
            go.Scatter(
                x=focus_rq["date_start"],
                y=focus_rq["recovery_pct"],
                mode="lines+markers",
                name="RQ direct DNB-BRDF",
                connectgaps=False,
                line=dict(color="#009227", width=2.8, shape="hv"),
            )
        )
        fig_gap_diagnostic.add_trace(
            go.Scatter(
                x=focus_gap_profile["date_start"],
                y=focus_gap_profile["recovery_pct"],
                mode="lines",
                name="Gap-filled diagnostic",
                line=dict(color="#D62728", width=2.0),
            )
        )
        fig_gap_diagnostic.add_hline(y=100, line_dash="dot", line_color="#7F8C8D")
        fig_gap_diagnostic.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line_dash="dash",
            line_color="#0057FF",
        )
        fig_gap_diagnostic.update_layout(
            template="plotly_white",
            paper_bgcolor="rgba(0,0,0,0)",
            title=f"Observed versus gap-filled recovery: {focus_row['unit_name']}",
            xaxis_title="Date",
            yaxis_title="Recovery relative to baseline (%)",
            width=1200,
            height=520,
            font=dict(family="Arial", size=13, color="#243B5A"),
            legend=dict(orientation="h", y=1.04),
        )
        fig_gap_diagnostic.show()
    except ValueError as error:
        print("Gap-filled diagnostic unavailable:", error)
else:
    print("Gap-filled band or unique focus municipality is unavailable; diagnostic skipped.")

## 9. Methods and provenance

The underlying workflow remains fully reproducible in the collapsed cells above:

- direct `DNB_BRDF_Corrected_NTL` with `MQF == 0`;
- GHSL G2 (codes 23 and 30), with G3 as the pre-declared coverage sensitivity;
- daily 95th-percentile spatial clamp;
- 60-day pre-Haiyan per-pixel median baseline;
- identical valid pixels in current and baseline sums;
- daily and non-overlapping four-day profiles;
- a 10% spatial-completeness gate inherited from the highest-ranked Samar–Leyte RQ configuration;
- two consecutive admissible four-day composites for T50, T80, and T90; and
- interval timing and baseline-IQR error bars rather than artificial continuity.

Key methodological sources are [Principe, Reinke, and Jones (2026)](https://doi.org/10.5194/isprs-archives-XLIX-B2-2026-1091-2026), [Román et al. (2019)](https://doi.org/10.1371/journal.pone.0218883), the [NASA Black Marble User Guide](https://viirsland.gsfc.nasa.gov/PDF/BlackMarbleUserGuide_v1.2_20210421.pdf), and [GHS-SMOD R2023A](https://doi.org/10.2905/A0DF7A6F-49DE-46EA-9BDE-563437A6E2BA).

In [ ]:
# ============================================================
# EXPORT STORY OUTPUTS
# ============================================================

municipal_daily.to_csv(OUTPUT_DIR / "municipal_daily_rq_profiles.csv", index=False)
municipal_four_day.to_csv(OUTPUT_DIR / "municipal_four_day_rq_profiles.csv", index=False)
municipal_metrics.to_csv(OUTPUT_DIR / "municipal_recovery_metrics.csv", index=False)
municipal_stage_summary.to_csv(OUTPUT_DIR / "municipal_stage_summary.csv", index=False)
municipal_reports.to_csv(OUTPUT_DIR / "municipal_profile_provenance.csv", index=False)
evidence_matrix.to_csv(OUTPUT_DIR / "local_evidence_matrix.csv", index=False)
bridge_summary.to_csv(OUTPUT_DIR / "regional_ngcp_bridge.csv", index=False)

if not poi_profiles.empty:
    poi_daily.to_csv(OUTPUT_DIR / "poi_daily_rq_profiles.csv", index=False)
    poi_four_day.to_csv(OUTPUT_DIR / "poi_four_day_rq_profiles.csv", index=False)
    poi_metrics.to_csv(OUTPUT_DIR / "poi_recovery_metrics.csv", index=False)
    poi_stage_summary.to_csv(OUTPUT_DIR / "poi_stage_summary.csv", index=False)
    poi_reports.to_csv(OUTPUT_DIR / "poi_profile_provenance.csv", index=False)

fig_locations.write_html(
    OUTPUT_DIR / "01_study_locations.html",
    include_plotlyjs="cdn",
)
fig_regional_recovery.write_html(
    OUTPUT_DIR / "02_regional_stage_recovery.html",
    include_plotlyjs="cdn",
)
fig_municipal_grid.write_html(
    OUTPUT_DIR / "03_municipality_profile_grid.html",
    include_plotlyjs="cdn",
)
fig_central_profile.write_html(
    OUTPUT_DIR / "04_centralized_profile.html",
    include_plotlyjs="cdn",
)
fig_impact.write_html(
    OUTPUT_DIR / "05_impact_drop.html",
    include_plotlyjs="cdn",
)
fig_milestones.write_html(
    OUTPUT_DIR / "06_recovery_milestones.html",
    include_plotlyjs="cdn",
)

for poi_name, figure in poi_story_figures.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", poi_name).strip("_").lower()
    figure.write_html(
        OUTPUT_DIR / f"poi_{safe_name}_story.html",
        include_plotlyjs="cdn",
    )

print("Story outputs written to:", OUTPUT_DIR)

## Takeaway

The local application is defensible only where the observation record supports it. The maps establish where the light changed; spatial completeness establishes whether the satellite actually observed the analytical support; the time series describes the functional trajectory; and T50, T80, and T90 are reported only after those conditions are satisfied.

**Regional alignment licenses the question, not the local answer.**